# Early Benzodiazepine Sedation, Incident Delirium, and Ventilator-Free Days

*A target-trial emulation with landmark causal mediation, built on MIMIC-IV v3.1*

---

### Research question
> *Among adults receiving early invasive ventilation and continuous sedation, does benzodiazepine use in the first 24 hours (versus propofol and/or dexmedetomidine without benzodiazepine) reduce ventilator-free days to day 28 - and how much of that effect is transmitted through incident CAM-ICU-positive delirium after a 48-hour landmark?*

### Why this mediation pathway should be real
Benzodiazepines are more deliriogenic than propofol or dexmedetomidine (PADIS). Delirium independently prolongs mechanical ventilation. The clinical implication is concrete: if a large share of the benzo-VFD association runs through delirium, delirium-prevention strategies (and benzo-sparing sedation) are mechanistic targets, not just correlated practices.

### Headline
The **primary story is the total effect**: early benzodiazepine sedation is associated with roughly **1.3 fewer ventilator-free days** to day 28. Incident delirium is a **small and assumption-sensitive secondary pathway** (proportion mediated ~7%, indirect-effect confidence interval crossing zero), not the confirmed mechanism. We therefore lead with the total effect and treat mediation as mechanistic, exploratory context.

### Design (target-trial emulation + landmark mediation)
Following a pre-specified target-trial protocol (Section 0.2, `tables/table0_protocol.csv`):
1. **Exposure (A), 0-24 h:** any midazolam or lorazepam infusion versus propofol/dexmedetomidine with no benzodiazepine, among early-ventilated adults who received continuous sedation, defined at time zero.
2. **Cohorts:** a **broad early-eligible cohort** (for the total effect) and a **48 h landmark subset** used only for mediation, reweighted back to the broad population with inverse-probability-of-selection weights.
3. **Mediator (M), after 48 h:** incident CAM-ICU Positive among patients free of Positive assessments at the landmark and assessed afterward.
4. **Outcome (Y):** ventilator-free days to day 28 (death within 28 days scores 0), with time to liberation and 28-day mortality as competing-risk companions.
5. **Confounding control:** MICE for missing baseline covariates (fit on the broad cohort); stabilized IPTW and overlap weights for treatment; inverse-probability-of-selection weights for landmark inclusion; and the full confounder set (including a `sofa_lite` severity score) entered symmetrically in the mediator and outcome models. A Love plot documents balance before vs after weighting.
6. **Estimand:** total effect (broad cohort) plus natural and interventional direct/indirect effects and proportion mediated (landmark cohort), estimated by weighted g-computation with an A x M interaction (Valeri and VanderWeele) and cross-checked with an inverse-odds-ratio mediator-weighting estimator; Rubin's-rules multiple-imputation inference; E-values for the total effect and the indirect effect.

> Every analytic choice lives in `CONFIG`. Figures use a vivid, publication-oriented palette.


## 0 - Causal framework and the mediation DAG

- **A:** early benzodiazepine strategy (drug-class choice in the first ICU day).
- **M:** incident delirium after the landmark (not a rephrasing of ventilation duration).
- **Y:** ventilator-free days (integrates duration of ventilation and early death).
- **C (baseline):** age, sex, race/ethnicity, admission type, insurance, ICU unit type, admission era (`anchor_year_group`), early severity (GCS, MAP, heart rate, lactate, creatinine, bilirubin, platelets), early vasopressor and opioid co-sedation, comorbidity burden, and a benzodiazepine-indication flag (alcohol/sedative use disorder, withdrawal, or seizure). The indication flag, unit, era, and comorbidity burden were added specifically to blunt confounding by indication - the dominant threat when the drug choice tracks the reason for admission. We do **not** adjust the primary models for post-treatment RASS depth, because RASS after sedative initiation sits on the pathway from drug choice.

Identification of natural effects requires no unmeasured confounding of A-Y, A-M, and M-Y, and no mediator-outcome confounder affected by A. Residual confounding (e.g. unmeasured indication for benzo) is probed with an E-value.

## 0.2 - Target-trial protocol and statistical analysis plan (pre-specified)

We emulate a target trial of early benzodiazepine sedation versus non-benzodiazepine sedation. The protocol is specified before any results, following the Hernan-Robins target-trial framework. A machine-readable copy is written to `tables/table0_protocol.csv`.

| Protocol element | Specification |
|---|---|
| **Eligibility** | Adults on their first ICU stay who receive invasive mechanical ventilation and continuous sedation within the first 24 h of ICU admission. Ascertained at time zero, not conditional on post-landmark survival. |
| **Treatment strategies** | (1) Early benzodiazepine: any continuous midazolam or lorazepam infusion in 0-24 h. (2) Non-benzodiazepine: propofol and/or dexmedetomidine in 0-24 h with no benzodiazepine. Opioid co-sedation is allowed in both arms and adjusted for. Post-24 h switching is not controlled (ITT-like grouping by initial strategy). |
| **Assignment** | Emulated randomization by stabilized IPTW on the full baseline confounder set; overlap weighting and unweighted analyses are reported as alternatives. Analysis is ITT-like (grouped by first-24 h strategy), not per-protocol. |
| **Time zero** | ICU admission with qualifying ventilation + sedation in the first 24 h. |
| **Landmark** | 48 h, used only to define incident (post-landmark) delirium and guarantee mediator temporality; sensitivity at 24 h and 72 h. Landmark restriction is handled with inverse-probability-of-selection weights so estimates target the broad early-eligible population. |
| **Follow-up / censoring** | 28 days from time zero. Competing event: in-hospital death (folded into VFD = 0 and modeled explicitly in cause-specific hazards). |
| **Causal contrasts** | Total effect (broad cohort); natural direct/indirect effects and interventional direct/indirect effects (landmark cohort) on the ventilator-free-days scale; proportion mediated by incident delirium. |
| **Primary analysis** | Multiply imputed (MICE) confounders; stabilized IPTW x selection weights; weighted g-computation for the natural-effect decomposition; Rubin's-rules inference (per-imputation bootstrap + between-imputation variance). |
| **Sensitivity hierarchy** | Midazolam-only exposure; exclude deep sedation (RASS < -4); exclude hard benzodiazepine indications; exclude CVICU; overlap weights; unweighted; landmark 24/48/72 h; sustained delirium (>= 2 positive CAM). |
| **Bias analyses** | E-value for the total effect and for the natural indirect effect (point and CI limit); competing-risk cause-specific hazards; 28-day mortality companion. |
| **Estimand summary** | On the VFD scale, how much of the effect of the early sedation strategy operates through incident delirium, in the broad early-eligible ventilated ICU population. |

**Discovery vs confirmation.** The primary estimand (NIE on VFD, IPTW x selection weights, 48 h landmark, full confounder set) is frozen. Dose-response, alternative landmarks, midazolam-versus-lorazepam contrasts, and effect-modifier analyses are exploratory and reported as secondary.


## Setup - environment and seed

In [ ]:
import os, sys, gc, warnings, math, re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap

from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge, LogisticRegression

import statsmodels.api as sm
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")
gc.enable()
pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 200)

RANDOM_STATE = 20260720
np.random.seed(RANDOM_STATE)

print("python      :", sys.version.split()[0])
print("pandas      :", pd.__version__)
print("numpy       :", np.__version__)
print("matplotlib  :", mpl.__version__)
import sklearn, statsmodels
print("scikit-learn:", sklearn.__version__)
print("statsmodels :", statsmodels.__version__)



## Setup - vivid publication palette

Semantic roles are fixed: **coral** for benzodiazepine exposure, **teal** for non-benzo sedation, **amber** for delirium (mediator), **cerulean** for the natural direct effect, **violet** for the natural indirect effect, **indigo** for the total effect.

In [ ]:
PALETTE = {
    "ink":       "#1B1919",
    "exposed":   "#AD002A",   # benzodiazepine
    "unexposed": "#00468B",   # propofol/dex, no benzo
    "mediator":  "#FDAF91",   # delirium
    "direct":    "#0099B4",
    "indirect":  "#925E9F",
    "total":     "#00468B",
    "accent":    "#ED0000",
    "sky":       "#0099B4",
    "orange":    "#FDAF91",
    "good":      "#42B540",
    "warn":      "#ED0000",
    "muted":     "#ADB6B6",
    "grid":      "#E6E9ED",
    "panel":     "#EFF3F6",
    "bg":        "#ffffff",
}
CAT_COLORS = ["#00468B", "#ED0000", "#42B540", "#0099B4", "#925E9F", "#FDAF91", "#AD002A", "#ADB6B6"]
CMAP_SEQ = LinearSegmentedColormap.from_list(
    "lancet_seq", ["#ffffff", "#D6E4EF", "#8FB9D6", "#0099B4", "#00468B", "#01254A"])
CMAP_DIV = LinearSegmentedColormap.from_list(
    "lancet_div", ["#AD002A", "#f4f4f4", "#00468B"])

mpl.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 300, "figure.facecolor": PALETTE["bg"],
    "savefig.facecolor": PALETTE["bg"], "savefig.bbox": "tight",
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif", "STIXGeneral"],
    "mathtext.fontset": "stix", "font.size": 11,
    "axes.titlesize": 14, "axes.titleweight": "bold", "axes.titlepad": 12,
    "axes.labelsize": 11.5, "axes.labelcolor": PALETTE["ink"], "axes.edgecolor": PALETTE["ink"],
    "axes.linewidth": 0.9, "axes.facecolor": PALETTE["bg"], "axes.grid": False,
    "grid.color": PALETTE["grid"], "grid.linewidth": 0.7,
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": PALETTE["ink"], "ytick.color": PALETTE["ink"],
    "text.color": PALETTE["ink"], "legend.frameon": False, "legend.fontsize": 10,
})

FIGDIR = Path("figures"); FIGDIR.mkdir(exist_ok=True)
TABDIR = Path("tables");  TABDIR.mkdir(exist_ok=True)

def save_fig(fig, name):
    fig.savefig(FIGDIR / f"{name}.png")
    fig.savefig(FIGDIR / f"{name}.pdf")
    try:
        fig.savefig(FIGDIR / f"{name}.tif", format="tiff", dpi=300,
                    pil_kwargs={"compression": "tiff_lzw"})
    except Exception as e:
        print(f"  (TIFF skipped: {e})")
    plt.close(fig)
    print(f"  saved -> figures/{name}.png | .pdf | .tif")

print("Palette ready. Figures -> ./figures , tables -> ./tables")

# Pre-specified target-trial protocol, written for the manuscript appendix.
protocol = pd.DataFrame([
    ("Eligibility", "Adults, first ICU stay, invasive ventilation + continuous sedation in first 24 h; ascertained at time zero"),
    ("Treatment strategies", "Early benzodiazepine (midazolam/lorazepam infusion 0-24 h) vs non-benzodiazepine (propofol/dexmedetomidine, no benzo); opioids allowed in both"),
    ("Assignment", "Emulated randomization via stabilized IPTW on the full confounder set (ITT-like); overlap/unweighted as alternatives"),
    ("Time zero", "ICU admission with qualifying ventilation + sedation in first 24 h"),
    ("Landmark", "48 h for incident-delirium temporality (24/72 h sensitivity); selection weighted to broad population"),
    ("Follow-up", "28 days; competing event in-hospital death (VFD=0 and cause-specific hazards)"),
    ("Causal contrasts", "Total effect (broad); natural + interventional direct/indirect effects (landmark); proportion mediated"),
    ("Primary analysis", "MICE + IPTW x selection weights + weighted g-computation; Rubin's-rules inference"),
    ("Sensitivity", "Midazolam-only; no deep sedation; exclude indication; exclude CVICU; overlap; unweighted; landmark 24/48/72; sustained delirium"),
    ("Bias analyses", "E-value (total and NIE, point + CI limit); cause-specific hazards; 28-day mortality companion"),
], columns=["Protocol element", "Specification"])
protocol.to_csv(TABDIR / "table0_protocol.csv", index=False)
print("Saved -> tables/table0_protocol.csv")


## Setup - paths and CONFIG

- `EXPOSURE_HOURS = 24` - benzodiazepine vs non-benzo sedation ascertained in the first ICU day.
- `LANDMARK_HOURS = 48` - mediator follow-up begins after this landmark (incident delirium only).
- `VFD_HORIZON_DAYS = 28` - ventilator-free days horizon.
- `MIN_IMPUTATIONS` / `MAX_IMPUTATIONS` - MICE completed datasets (rule-of-thumb capped).
- `N_BOOT` - nonparametric bootstrap resamples for mediation CIs (re-estimates PS + weights each draw).


In [ ]:
_CANDIDATE_ROOTS = [
    Path("physionet.org/files/mimiciv/3.1"),
    Path("../physionet.org/files/mimiciv/3.1"),
]
DATA_ROOT = next((p for p in _CANDIDATE_ROOTS if p.exists()), _CANDIDATE_ROOTS[0])
HOSP, ICU = DATA_ROOT / "hosp", DATA_ROOT / "icu"

CONFIG = {
    "EXPOSURE_HOURS": 24,
    "LANDMARK_HOURS": 48,
    "VFD_HORIZON_DAYS": 28,
    "MIN_AGE": 18,
    "N_BOOT": 150,                 # keep moderate to limit RAM/CPU; raise after a clean run
    "MIN_IMPUTATIONS": 5,
    "MAX_IMPUTATIONS": 10,         # cap MICE copies held in memory
    "MISSING_DROP_THRESHOLD": 0.40,
    "CHUNKSIZE": 1_000_000,        # smaller chunks = lower peak RAM
    "DEV_SAMPLE_STAYS": None,      # e.g. 8000 for a fast dry-run
}

# Itemids
CAM_ITEM, RASS_ITEM = 228332, 228096
MIDAZ, LORA = 221668, 221385
PROP, DEX = 222168, {225150, 229420}
IMV, EXTUB = 225792, 227194
NOREPI = 221906
VASO_SET = {221906, 221289, 222315, 221749, 221662}
# Continuous opioid co-sedation (fentanyl variants, morphine, hydromorphone)
OPIOID_SET = {221744, 225942, 225154, 221833}
CE_VITALS = {
    220045: "hr",
    220052: "map", 220181: "map",
    220739: "gcs_eye", 223900: "gcs_verbal", 223901: "gcs_motor",
    226512: "weight", 224639: "weight",
}
LAB = {"creatinine": 50912, "lactate": 50813, "bilirubin": 50885, "platelets": 51265}

REQUIRED = {
    "icustays": ICU / "icustays.csv.gz",
    "admissions": HOSP / "admissions.csv.gz",
    "patients": HOSP / "patients.csv.gz",
    "chartevents": ICU / "chartevents.csv.gz",
    "inputevents": ICU / "inputevents.csv.gz",
    "procedureevents": ICU / "procedureevents.csv.gz",
    "labevents": HOSP / "labevents.csv.gz",
    "diagnoses_icd": HOSP / "diagnoses_icd.csv.gz",
}
print("DATA_ROOT:", DATA_ROOT.resolve())
for k, p in REQUIRED.items():
    flag = "OK" if p.exists() else "MISSING"
    print(f"  [{flag}] {k:16s} {p}")


## 0.1 - Render the mediation DAG

In [ ]:
def draw_dag():
    fig, ax = plt.subplots(figsize=(12, 6.4))
    ax.set_xlim(0, 12); ax.set_ylim(0, 7); ax.axis("off")

    def node(x, y, label, w, h, fc, ec, emph=False, tcol="white"):
        ax.add_patch(FancyBboxPatch((x - w/2, y - h/2), w, h,
                     boxstyle="round,pad=0.02,rounding_size=0.12",
                     fc=fc, ec=ec, lw=2.2 if emph else 1.3, zorder=3))
        ax.text(x, y, label, ha="center", va="center",
                fontsize=10.5 if emph else 9.2,
                fontweight="bold" if emph else "normal", color=tcol, zorder=4)
        return {"x": x, "y": y, "w": w, "h": h}

    def rim(n, tx, ty):
        ang = np.arctan2(ty - n["y"], tx - n["x"])
        return (n["x"] + (n["w"]/2) * np.cos(ang), n["y"] + (n["h"]/2) * np.sin(ang))

    def link(n1, n2, col, lw=1.6, rad=0.0):
        sx, sy = rim(n1, n2["x"], n2["y"]); ex, ey = rim(n2, n1["x"], n1["y"])
        ax.add_patch(FancyArrowPatch((sx, sy), (ex, ey),
                     connectionstyle=f"arc3,rad={rad}", arrowstyle="-|>",
                     mutation_scale=15, lw=lw, color=col, zorder=2))

    C = node(1.7, 3.5,
             "Baseline confounders (C)\nage, sex, race, admission,\nICU unit, era, severity (GCS,\nMAP, HR, lactate, creatinine,\nbilirubin, platelets, vasopressors),\ncomorbidity burden, benzo\nindication (alcohol/sedative/\nseizure), early opioids",
             3.2, 3.0, PALETTE["panel"], PALETTE["muted"], tcol=PALETTE["ink"])
    A = node(6.0, 5.4, "Early benzo\nsedation  (A)\n0-24 h", 2.2, 1.15, PALETTE["exposed"], PALETTE["exposed"], emph=True)
    M = node(6.0, 1.9, "Incident delirium\nafter 48 h  (M)", 2.2, 1.1, PALETTE["mediator"], PALETTE["mediator"], emph=True, tcol=PALETTE["ink"])
    Y = node(10.3, 3.7, "Ventilator-free\ndays to day 28\n(Y)", 2.2, 1.25, PALETTE["total"], PALETTE["total"], emph=True)

    for n in (A, M, Y):
        link(C, n, PALETTE["muted"], lw=1.0, rad=0.05)
    link(A, M, PALETTE["indirect"], lw=3.0)
    link(M, Y, PALETTE["indirect"], lw=3.0)
    link(A, Y, PALETTE["direct"], lw=3.0, rad=-0.12)

    ax.text(4.5, 3.7, "indirect\n(A->M->Y)", color=PALETTE["indirect"],
            fontsize=10, fontstyle="italic", ha="center", fontweight="bold")
    ax.text(8.5, 5.15, "direct (A->Y)", color=PALETTE["direct"],
            fontsize=10, fontstyle="italic", ha="center", fontweight="bold")
    ax.text(6.0, 0.3,
            "Colored arrows: decomposition of interest. Gray: confounding, removed by conditioning on C.",
            ha="center", fontsize=8.8, color=PALETTE["ink"])
    save_fig(fig, "fig00_dag"); plt.show()

draw_dag()

## 1 - Load dimension tables and build the first-ICU-stay base cohort

In [ ]:
def read_gz(path, usecols, parse_dates=None, dtype=None):
    df = pd.read_csv(path, compression="gzip", usecols=usecols, dtype=dtype, parse_dates=parse_dates)
    print(f"  loaded {path.name:28s} rows={len(df):>10,d}")
    return df

icustays = read_gz(REQUIRED["icustays"],
    usecols=["subject_id", "hadm_id", "stay_id", "intime", "outtime", "los", "first_careunit"],
    parse_dates=["intime", "outtime"],
    dtype={"subject_id": "int64", "hadm_id": "int64", "stay_id": "int64", "los": "float32"})
admissions = read_gz(REQUIRED["admissions"],
    usecols=["subject_id", "hadm_id", "admittime", "dischtime", "deathtime",
             "admission_type", "insurance", "race", "hospital_expire_flag"],
    parse_dates=["admittime", "dischtime", "deathtime"],
    dtype={"subject_id": "int64", "hadm_id": "int64", "hospital_expire_flag": "int8"})
patients = read_gz(REQUIRED["patients"],
    usecols=["subject_id", "gender", "anchor_age", "anchor_year_group", "dod"],
    parse_dates=["dod"],
    dtype={"subject_id": "int64", "anchor_age": "int16"})

icu_sorted = icustays.sort_values(["subject_id", "intime"])
first_icu = icu_sorted.groupby("subject_id", as_index=False).first()
cohort = (first_icu
          .merge(patients, on="subject_id", how="left")
          .merge(admissions, on=["subject_id", "hadm_id"], how="left"))
cohort = cohort[cohort["anchor_age"] >= CONFIG["MIN_AGE"]].copy()

if CONFIG["DEV_SAMPLE_STAYS"] is not None and len(cohort) > CONFIG["DEV_SAMPLE_STAYS"]:
    cohort = cohort.sample(CONFIG["DEV_SAMPLE_STAYS"], random_state=RANDOM_STATE).reset_index(drop=True)
    print(f"DEV sub-sample: {len(cohort):,d}")

COHORT_STAYS = set(cohort["stay_id"].tolist())

# Drop dimension tables once merged (free RAM for later scans)
del icustays, admissions, patients, icu_sorted, first_icu
gc.collect()
HADMS = set(cohort["hadm_id"].dropna().astype("int64").tolist())
print(f"Base first adult ICU stays: {len(cohort):,d}")

# ---- Diagnoses: benzodiazepine indication + comorbidity burden -------------
# ICD-9/10 prefix maps. "Indication" captures the clinical reasons a clinician
# preferentially reaches for a benzodiazepine (alcohol/sedative use disorder,
# withdrawal, seizure/status epilepticus) - the core confounder-by-indication.
IND_PREFIX = (
    "2910", "2918", "2913", "29181", "3039", "3050", "30500",  # ICD-9 alcohol
    "2920", "3040", "3041", "3053", "3054",                     # ICD-9 sedative/drug
    "3450", "3451", "3452", "3453", "3454", "3455", "3457", "3459", "78039",  # ICD-9 seizure
    "F10", "F13",                                                # ICD-10 alcohol/sedative
    "G40", "G41", "R560",                                        # ICD-10 seizure
)
# Elixhauser-lite comorbidity categories (prefix -> count of distinct groups).
COMORB = {
    "chf":       (("428",), ("I50",)),
    "arrhythmia":(("427",), ("I47", "I48", "I49")),
    "pulm":      (("490", "491", "492", "493", "494", "495", "496"), ("J40", "J41", "J42", "J43", "J44", "J45", "J47")),
    "diabetes":  (("250",), ("E10", "E11", "E13")),
    "renal":     (("585", "586"), ("N18", "N19")),
    "liver":     (("571", "5722", "5723", "5724", "5728"), ("K70", "K72", "K74")),
    "cancer":    (("140", "141", "142", "150", "151", "162", "174", "185"), ("C",)),
    "metastatic":(("196", "197", "198", "199"), ("C77", "C78", "C79", "C80")),
    "cvd":       (("430", "431", "432", "433", "434", "436"), ("I60", "I61", "I62", "I63", "I64")),
    "hypertension":(("401", "402", "403", "404", "405"), ("I10", "I11", "I12", "I13", "I15")),
    "cad":       (("410", "411", "412", "413", "414"), ("I20", "I21", "I22", "I24", "I25")),
    "obesity":   (("2780",), ("E66",)),
}
DX_CACHE = TABDIR / "_dx_cache.csv"
if DX_CACHE.exists():
    dx = pd.read_csv(DX_CACHE)
    print(f"Loaded diagnoses cache: {len(dx):,d} admissions")
else:
    _ind_hadm = set()
    _cm_hadm = {k: set() for k in COMORB}
    for chunk in pd.read_csv(
            REQUIRED["diagnoses_icd"], compression="gzip",
            usecols=["hadm_id", "icd_code", "icd_version"],
            chunksize=CONFIG["CHUNKSIZE"],
            dtype={"hadm_id": "int64", "icd_code": "string", "icd_version": "int8"}):
        chunk = chunk[chunk["hadm_id"].isin(HADMS)].dropna(subset=["icd_code"])
        if chunk.empty:
            continue
        code = chunk["icd_code"].str.replace(".", "", regex=False).str.upper().str.strip()
        hid = chunk["hadm_id"].to_numpy()
        ind_mask = code.str.startswith(IND_PREFIX)
        _ind_hadm.update(hid[ind_mask.to_numpy()].tolist())
        for grp, (p9, p10) in COMORB.items():
            m = code.str.startswith(tuple(p9) + tuple(p10))
            _cm_hadm[grp].update(hid[m.to_numpy()].tolist())
        del chunk
    gc.collect()
    hlist = sorted(HADMS)
    dx = pd.DataFrame({"hadm_id": hlist})
    dx["sud_indication"] = dx["hadm_id"].isin(_ind_hadm).astype("int8")
    cm = pd.Series(0, index=dx.index, dtype="int16")
    for grp, hset in _cm_hadm.items():
        cm = cm + dx["hadm_id"].isin(hset).astype("int16")
    dx["comorbidity_count"] = cm.astype("float32")
    dx.to_csv(DX_CACHE, index=False)
    print(f"Built diagnoses cache -> {DX_CACHE.name}: {len(dx):,d} admissions")
    del _ind_hadm, _cm_hadm; gc.collect()

cohort = cohort.merge(dx, on="hadm_id", how="left")
cohort["sud_indication"] = cohort["sud_indication"].fillna(0).astype("int8")
cohort["comorbidity_count"] = cohort["comorbidity_count"].fillna(0).astype("float32")
print(f"Benzo indication (alcohol/sedative/seizure Dx): {int(cohort['sud_indication'].sum()):,d}")
print(f"Comorbidity count: mean={cohort['comorbidity_count'].mean():.2f} "
      f"max={int(cohort['comorbidity_count'].max())}")


## 2 - Early invasive ventilation and ventilator-free days (procedureevents)

Invasive ventilation is identified from `procedureevents` itemid 225792. Early IMV requires a ventilation interval overlapping the first 24 h. Ventilator-free days to day 28 use the standard convention: **0 if death occurs within 28 days**, otherwise `28 - days of IMV` in the first 28 days after ICU admission.

In [ ]:
EXP_H = CONFIG["EXPOSURE_HOURS"]
LAND_H = CONFIG["LANDMARK_HOURS"]
HORIZON = CONFIG["VFD_HORIZON_DAYS"]

stay_t0 = cohort.set_index("stay_id")["intime"]
vent_hours = {}
early_imv_set = set()

for chunk in pd.read_csv(
        REQUIRED["procedureevents"], compression="gzip",
        usecols=["stay_id", "itemid", "starttime", "endtime"],
        parse_dates=["starttime", "endtime"], chunksize=CONFIG["CHUNKSIZE"],
        dtype={"stay_id": "int64", "itemid": "int64"}):
    chunk = chunk[chunk["stay_id"].isin(COHORT_STAYS) & (chunk["itemid"] == IMV)]
    if chunk.empty:
        continue
    chunk = chunk.copy()
    chunk["t0"] = chunk["stay_id"].map(stay_t0)
    chunk = chunk.dropna(subset=["t0", "starttime"])
    chunk["endtime"] = chunk["endtime"].fillna(chunk["starttime"] + pd.Timedelta(hours=1))
    chunk["win_end"] = chunk["t0"] + pd.Timedelta(days=HORIZON)
    chunk["s"] = chunk[["starttime", "t0"]].max(axis=1)
    chunk["e"] = chunk[["endtime", "win_end"]].min(axis=1)
    chunk = chunk[chunk["e"] > chunk["s"]]
    if chunk.empty:
        continue
    chunk["hours"] = (chunk["e"] - chunk["s"]).dt.total_seconds() / 3600.0
    early = (chunk["starttime"] < chunk["t0"] + pd.Timedelta(hours=EXP_H)) & (chunk["endtime"] > chunk["t0"])
    for sid, h in chunk.groupby("stay_id")["hours"].sum().items():
        vent_hours[sid] = vent_hours.get(sid, 0.0) + float(h)
    early_imv_set.update(chunk.loc[early, "stay_id"].tolist())
    del chunk
gc.collect()

cohort["early_imv"] = cohort["stay_id"].isin(early_imv_set)
cohort["vent_hours_28d"] = cohort["stay_id"].map(vent_hours).fillna(0.0).astype("float32")
cohort["vent_days_28d"] = (cohort["vent_hours_28d"] / 24.0).clip(upper=HORIZON)

t0 = cohort["intime"]
death_any = cohort["deathtime"].fillna(cohort["dod"])
cohort["dead_28d"] = ((death_any.notna()) & (death_any <= t0 + pd.Timedelta(days=HORIZON))).astype("int8")
cohort["vfd_28"] = np.where(
    cohort["dead_28d"] == 1, 0.0, (HORIZON - cohort["vent_days_28d"]).clip(lower=0)).astype("float32")

# Days from ICU admission to death (capped at horizon) for competing-risk models
_dd = (death_any - t0).dt.total_seconds() / 86400.0
cohort["days_to_death"] = _dd.where(_dd.notna(), np.inf).clip(lower=0)

print(f"Early IMV stays : {int(cohort['early_imv'].sum()):,d}")
print(f"VFD-28 summary  : median={cohort.loc[cohort['early_imv'], 'vfd_28'].median():.1f}")
del vent_hours, early_imv_set; gc.collect()


## 3 - Early sedative exposure from inputevents (benzodiazepine vs non-benzo)

Among patients with any continuous sedation (midazolam, lorazepam, propofol, or dexmedetomidine) started in the first 24 h:
- **A = 1** if any benzodiazepine (midazolam or lorazepam) was infused
- **A = 0** if propofol and/or dexmedetomidine was used **without** benzodiazepine

In [ ]:
INP_CACHE = TABDIR / "_inp_cache.csv"
FLAG_COLS = ["early_midaz", "early_lora", "early_prop", "early_dex", "early_vaso", "early_opioid"]
if INP_CACHE.exists():
    inp = pd.read_csv(INP_CACHE)
    print(f"Loaded inputevents cache: {len(inp):,d} stays")
else:
    sed_ids = {MIDAZ, LORA, PROP} | set(DEX)
    keep_items = sed_ids | VASO_SET | OPIOID_SET
    flags = {"midaz": set(), "lora": set(), "prop": set(), "dex": set(), "vaso": set(), "opioid": set()}
    t0_map = stay_t0  # from prior cell
    for chunk in pd.read_csv(
            REQUIRED["inputevents"], compression="gzip",
            usecols=["stay_id", "itemid", "starttime"],
            parse_dates=["starttime"], chunksize=CONFIG["CHUNKSIZE"],
            dtype={"stay_id": "int64", "itemid": "int64"}):
        chunk = chunk[chunk["stay_id"].isin(COHORT_STAYS) & chunk["itemid"].isin(keep_items)]
        if chunk.empty:
            continue
        chunk = chunk.copy()
        chunk["t0"] = chunk["stay_id"].map(t0_map)
        chunk = chunk.dropna(subset=["t0", "starttime"])
        lo = chunk["t0"] - pd.Timedelta(hours=2)
        hi = chunk["t0"] + pd.Timedelta(hours=EXP_H)
        chunk = chunk[(chunk["starttime"] >= lo) & (chunk["starttime"] < hi)]
        if chunk.empty:
            continue
        flags["midaz"].update(chunk.loc[chunk["itemid"] == MIDAZ, "stay_id"])
        flags["lora"].update(chunk.loc[chunk["itemid"] == LORA, "stay_id"])
        flags["prop"].update(chunk.loc[chunk["itemid"] == PROP, "stay_id"])
        flags["dex"].update(chunk.loc[chunk["itemid"].isin(DEX), "stay_id"])
        flags["vaso"].update(chunk.loc[chunk["itemid"].isin(VASO_SET), "stay_id"])
        flags["opioid"].update(chunk.loc[chunk["itemid"].isin(OPIOID_SET), "stay_id"])
        del chunk
    gc.collect()
    inp = pd.DataFrame({"stay_id": sorted(COHORT_STAYS)})
    for col, key in zip(FLAG_COLS, ["midaz", "lora", "prop", "dex", "vaso", "opioid"]):
        inp[col] = inp["stay_id"].isin(flags[key]).astype("int8")
    inp.to_csv(INP_CACHE, index=False)
    print(f"Built inputevents cache -> {INP_CACHE.name}: {len(inp):,d} stays")
    del flags; gc.collect()

cohort = cohort.merge(inp, on="stay_id", how="left")
for col in FLAG_COLS:
    cohort[col] = cohort[col].fillna(0).astype(bool)
cohort["early_benzo"] = cohort["early_midaz"] | cohort["early_lora"]
cohort["early_nonbenzo_sed"] = cohort["early_prop"] | cohort["early_dex"]
cohort["early_any_sed"] = cohort["early_benzo"] | cohort["early_nonbenzo_sed"]

cohort["A"] = np.where(
    cohort["early_benzo"], 1,
    np.where(cohort["early_nonbenzo_sed"] & ~cohort["early_benzo"], 0, np.nan))

print(cohort[["early_benzo", "early_prop", "early_dex", "early_any_sed"]].sum())
print("A distribution among non-missing:", cohort["A"].value_counts(dropna=True).to_dict())
gc.collect()


## 4 - Single chartevents pass: CAM-ICU, RASS, and early vitals

We stream `chartevents` once and **aggregate on the fly** (sets / running means) so peak RAM stays bounded. We never materialize the full CAM/RASS/vital event tables.


In [ ]:
# Single chartevents pass -> per-stay scalars cached to disk. We store CAM-ICU
# event TIMING (hours from ICU intime) rather than pre/post-48h buckets, so the
# landmark (24/48/72 h) and delirium-misclassification sensitivities can be
# derived downstream WITHOUT re-streaming. Delete tables/_ce_cache.csv to rebuild.
CE_CACHE = TABDIR / "_ce_cache.csv"
EXP_NS = np.timedelta64(EXP_H, "h")
PRE_NS = np.timedelta64(6, "h")
HORIZON_NS = np.timedelta64(HORIZON + 1, "D")

if CE_CACHE.exists():
    ce_cache = pd.read_csv(CE_CACHE)
    print(f"Loaded chartevents cache: {len(ce_cache):,d} stays (delete {CE_CACHE.name} to rebuild)")
else:
    stay_intime = cohort.set_index("stay_id")["intime"]
    ce_keep = {CAM_ITEM, RASS_ITEM} | set(CE_VITALS)

    cam_pos_times = {}     # stay -> sorted list of positive-CAM hours
    cam_assess_lo = {}     # stay -> earliest CAM assessment hour
    cam_assess_hi = {}     # stay -> latest CAM assessment hour
    cam_n_assess = {}      # stay -> count of CAM assessments
    rass_sum, rass_n = {}, {}
    vit_sum, vit_n = {}, {}

    def _parse_rass(val, num):
        if pd.notna(num):
            try:
                return float(num)
            except Exception:
                pass
        m = re.match(r"^\s*(-?\d+)", str(val))
        return float(m.group(1)) if m else np.nan

    n_seen = 0
    for chunk in pd.read_csv(
            REQUIRED["chartevents"], compression="gzip",
            usecols=["stay_id", "charttime", "itemid", "value", "valuenum"],
            parse_dates=["charttime"], chunksize=CONFIG["CHUNKSIZE"],
            dtype={"stay_id": "int64", "itemid": "int64", "valuenum": "float32"}):
        chunk = chunk[chunk["stay_id"].isin(COHORT_STAYS) & chunk["itemid"].isin(ce_keep)]
        n_seen += CONFIG["CHUNKSIZE"]
        if chunk.empty:
            continue
        chunk = chunk.copy()
        chunk["t0"] = chunk["stay_id"].map(stay_intime)
        chunk["dt"] = chunk["charttime"] - chunk["t0"]
        chunk = chunk[(chunk["dt"] > -PRE_NS) & (chunk["dt"] <= HORIZON_NS)]
        if chunk.empty:
            continue
        chunk["dt_h"] = chunk["dt"].dt.total_seconds() / 3600.0

        cam = chunk[chunk["itemid"] == CAM_ITEM]
        if not cam.empty:
            val = cam["value"].astype(str).str.strip().str.lower()
            pos = val.str.contains("positive", na=False)
            for sid, h in zip(cam["stay_id"].to_numpy(), cam["dt_h"].to_numpy()):
                sid = int(sid)
                cam_assess_lo[sid] = min(cam_assess_lo.get(sid, np.inf), float(h))
                cam_assess_hi[sid] = max(cam_assess_hi.get(sid, -np.inf), float(h))
                cam_n_assess[sid] = cam_n_assess.get(sid, 0) + 1
            for sid, h in zip(cam.loc[pos, "stay_id"].to_numpy(), cam.loc[pos, "dt_h"].to_numpy()):
                cam_pos_times.setdefault(int(sid), []).append(float(h))

        rass = chunk[(chunk["itemid"] == RASS_ITEM) & (chunk["dt"] <= EXP_NS)]
        if not rass.empty:
            for sid, v, num in zip(rass["stay_id"].to_numpy(), rass["value"].to_numpy(), rass["valuenum"].to_numpy()):
                sc = _parse_rass(v, num)
                if sc != sc:
                    continue
                rass_sum[sid] = rass_sum.get(sid, 0.0) + sc
                rass_n[sid] = rass_n.get(sid, 0) + 1

        vit = chunk[chunk["itemid"].isin(CE_VITALS) & (chunk["dt"] <= EXP_NS)]
        if not vit.empty:
            vit = vit.dropna(subset=["valuenum"])
            metrics = vit["itemid"].map(CE_VITALS)
            for sid, met, val in zip(vit["stay_id"].to_numpy(), metrics.to_numpy(), vit["valuenum"].to_numpy()):
                key = (int(sid), met)
                vit_sum[key] = vit_sum.get(key, 0.0) + float(val)
                vit_n[key] = vit_n.get(key, 0) + 1

        del chunk, cam, rass, vit
        if n_seen % 20_000_000 == 0:
            print(f"  ... scanned ~{n_seen/1e6:.0f}M chartevents rows")
            gc.collect()

    def _nth_pos(sid, n):
        t = sorted(cam_pos_times.get(sid, []))
        return t[n] if len(t) > n else np.nan

    rows = []
    for sid in COHORT_STAYS:
        vit_of = lambda m: (vit_sum[(sid, m)] / vit_n[(sid, m)]) if (sid, m) in vit_n else np.nan
        rows.append({
            "stay_id": sid,
            "cam_first_pos_h": _nth_pos(sid, 0),
            "cam_second_pos_h": _nth_pos(sid, 1),
            "cam_n_pos": len(cam_pos_times.get(sid, [])),
            "cam_first_assess_h": cam_assess_lo.get(sid, np.nan),
            "cam_last_assess_h": cam_assess_hi.get(sid, np.nan),
            "cam_n_assess": cam_n_assess.get(sid, 0),
            "rass_mean_24h": (rass_sum[sid] / rass_n[sid]) if sid in rass_n else np.nan,
            "hr": vit_of("hr"), "map": vit_of("map"),
            "gcs_eye": vit_of("gcs_eye"), "gcs_verbal": vit_of("gcs_verbal"),
            "gcs_motor": vit_of("gcs_motor"), "weight": vit_of("weight"),
        })
    ce_cache = pd.DataFrame(rows)
    ce_cache.to_csv(CE_CACHE, index=False)
    print(f"Built chartevents cache -> {CE_CACHE.name}: {len(ce_cache):,d} stays")
    del cam_pos_times, cam_assess_lo, cam_assess_hi, cam_n_assess
    del rass_sum, rass_n, vit_sum, vit_n
    gc.collect()


## 4.1 - Define the mediator: incident CAM-ICU-positive delirium after 48 h

In [ ]:
# Merge cached per-stay scalars, then derive early vitals and RASS.
cohort = cohort.merge(ce_cache, on="stay_id", how="left")
cohort["gcs"] = cohort[["gcs_eye", "gcs_verbal", "gcs_motor"]].sum(axis=1, min_count=1).astype("float32")
for met in ["hr", "map", "weight", "rass_mean_24h"]:
    cohort[met] = cohort[met].astype("float32")

def derive_delirium(df, land_h, require_sustained=False):
    """Landmark-parameterized mediator flags from cached CAM timing.

    prevalent  : first positive CAM at/before the landmark
    incident(M): first positive CAM strictly after the landmark (delirium-free at landmark)
    require_sustained: incident only if >= 2 positive CAMs (misclassification sensitivity)
    """
    fp = df["cam_first_pos_h"]
    prevalent = (fp <= land_h)
    incident = (fp > land_h) & df["cam_first_pos_h"].notna()
    if require_sustained:
        incident = incident & (df["cam_n_pos"] >= 2)
    assessed_pre = (df["cam_first_assess_h"] <= land_h)
    assessed_post = (df["cam_last_assess_h"] > land_h)
    assessed_ever = (df["cam_n_assess"] > 0)
    return (prevalent.fillna(False).astype("int8"),
            incident.fillna(False).astype("int8"),
            assessed_pre.fillna(False).astype("int8"),
            assessed_post.fillna(False).astype("int8"),
            assessed_ever.fillna(False).astype("int8"))

(cohort["prevalent_delirium"], cohort["M"], cohort["assessed_pre"],
 cohort["assessed_post"], cohort["assessed_ever"]) = derive_delirium(cohort, LAND_H)

print(f"Prevalent delirium : {int(cohort['prevalent_delirium'].sum()):,d}")
print(f"Incident delirium  : {int(cohort['M'].sum()):,d}")
print(f"Assessed post-landmark: {int(cohort['assessed_post'].sum()):,d}")
gc.collect()


## 4.2 - Early labs (creatinine, lactate) from labevents

In [ ]:
LAB_CACHE = TABDIR / "_lab_cache.csv"
if LAB_CACHE.exists():
    labdf = pd.read_csv(LAB_CACHE)
    print(f"Loaded labevents cache: {len(labdf):,d} admissions")
else:
    adm_t0 = cohort.dropna(subset=["hadm_id"]).drop_duplicates("hadm_id").set_index("hadm_id")["intime"]
    lab_ids = set(LAB.values())
    lab_map = {v: k for k, v in LAB.items()}
    lab_sum, lab_n = {}, {}
    for chunk in pd.read_csv(
            REQUIRED["labevents"], compression="gzip",
            usecols=["hadm_id", "itemid", "charttime", "valuenum"],
            parse_dates=["charttime"], chunksize=CONFIG["CHUNKSIZE"],
            dtype={"hadm_id": "Int64", "itemid": "int64", "valuenum": "float32"}):
        chunk = chunk.dropna(subset=["hadm_id", "valuenum"])
        chunk = chunk[chunk["hadm_id"].isin(HADMS) & chunk["itemid"].isin(lab_ids)]
        if chunk.empty:
            continue
        chunk = chunk.copy()
        chunk["t0"] = chunk["hadm_id"].map(adm_t0)
        chunk = chunk.dropna(subset=["t0"])
        chunk["dt"] = chunk["charttime"] - chunk["t0"]
        chunk = chunk[(chunk["dt"] > -PRE_NS) & (chunk["dt"] <= EXP_NS)]
        if chunk.empty:
            continue
        chunk["metric"] = chunk["itemid"].map(lab_map)
        for hid, met, val in zip(chunk["hadm_id"].astype("int64").to_numpy(),
                                 chunk["metric"].to_numpy(),
                                 chunk["valuenum"].to_numpy()):
            key = (int(hid), met)
            lab_sum[key] = lab_sum.get(key, 0.0) + float(val)
            lab_n[key] = lab_n.get(key, 0) + 1
        del chunk
    gc.collect()
    labdf = pd.DataFrame({"hadm_id": sorted(HADMS)})
    for met in LAB:
        series = {hid: lab_sum[(hid, met)] / lab_n[(hid, met)]
                  for (hid, m) in lab_n if m == met}
        labdf[met] = labdf["hadm_id"].map(series).astype("float32")
    labdf.to_csv(LAB_CACHE, index=False)
    print(f"Built labevents cache -> {LAB_CACHE.name}: {len(labdf):,d} admissions")
    del lab_sum, lab_n, adm_t0; gc.collect()

cohort = cohort.merge(labdf, on="hadm_id", how="left")
for met in LAB:
    cohort[met] = cohort[met].astype("float32")
print(cohort[["creatinine", "lactate", "bilirubin", "platelets"]].describe().round(2))
gc.collect()


## 5 - Eligibility and the analytic cohort

In [ ]:
# ---- Target-trial eligibility: a BROAD early-eligible cohort defined at time
#      zero (first 24 h of ventilation + continuous sedation), and a LANDMARK
#      subset used only for mediation. The total effect is estimated in the broad
#      cohort; the landmark restriction (survive + CAM-assessable past 48 h,
#      delirium-free at landmark) is handled with selection weights downstream.
flow = []
def step(df, label):
    flow.append((label, len(df)))
    return df

c = step(cohort, "First adult ICU stays")
c = step(c[c["early_imv"]], "Early invasive mechanical ventilation (first 24 h)")
c = step(c[c["early_any_sed"]], "Continuous sedation in first 24 h (benzo/propofol/dex)")
c = step(c[c["A"].notna()], "Classifiable exposure (benzo vs non-benzo-only)")
broad = step(c[c["vfd_28"].notna()], "Ventilator-free days defined [BROAD target-trial cohort]").copy()

# Landmark restriction (mediation cohort only)
lm = broad[broad["assessed_ever"] == 1]
lm = step(lm, "CAM-ICU protocol used at least once")
lm = step(lm[lm["prevalent_delirium"] == 0], "No prevalent CAM-positive delirium at/before landmark")
lm = step(lm[lm["assessed_post"] == 1], f"CAM assessed after the {LAND_H} h landmark [LANDMARK cohort]")

def collapse_race(x):
    s = str(x).upper()
    if "WHITE" in s: return "White"
    if "BLACK" in s: return "Black"
    if "HISPANIC" in s or "LATINO" in s: return "Hispanic"
    if "ASIAN" in s: return "Asian"
    return "Other/Unknown"

def collapse_unit(x):
    s = str(x).upper()
    if "MICU" in s or "MEDICAL" in s: return "MICU"
    if "SICU" in s or "SURG" in s or "TRAUMA" in s: return "SICU"
    if "CVICU" in s or "CARDIAC VASCULAR" in s or "CSRU" in s: return "CVICU"
    if "CCU" in s or "CORONARY" in s: return "CCU"
    if "NEURO" in s: return "Neuro"
    return "Other"

def sofa_lite(df):
    """Admission SOFA-like severity (0-19) from measured components. Respiratory
    (PaO2/FiO2) is unavailable in this extract, so this is a 5-system proxy
    (CNS, cardiovascular, renal, coagulation, liver). Missing components score 0."""
    gcs = df["gcs"]
    cns = np.select([gcs < 6, gcs < 10, gcs < 13, gcs < 15], [4, 3, 2, 1], default=0)
    cardio = np.where(df["early_vaso"].astype(float) == 1, 3,
                      np.where(df["map"] < 70, 1, 0))
    cr = df["creatinine"]
    renal = np.select([cr >= 5, cr >= 3.5, cr >= 2.0, cr >= 1.2], [4, 3, 2, 1], default=0)
    plt_ = df["platelets"]
    coag = np.select([plt_ < 20, plt_ < 50, plt_ < 100, plt_ < 150], [4, 3, 2, 1], default=0)
    bl = df["bilirubin"]
    liver = np.select([bl >= 12, bl >= 6, bl >= 2, bl >= 1.2], [4, 3, 2, 1], default=0)
    for arr, src in [(cns, gcs), (renal, cr), (coag, plt_), (liver, bl)]:
        arr[src.isna().to_numpy()] = 0
    return (cns + cardio + renal + coag + liver).astype("float32")

def add_covariates(df):
    df = df.copy()
    df["female"] = (df["gender"].astype(str).str.upper() == "F").astype(int)
    df["race_eth"] = pd.Categorical(df["race"].map(collapse_race),
        categories=["White", "Black", "Hispanic", "Asian", "Other/Unknown"])
    df["admission_type"] = df["admission_type"].astype("category")
    df["insurance"] = df["insurance"].astype("category")
    df["age"] = df["anchor_age"].astype(float)
    df["careunit_grp"] = pd.Categorical(df["first_careunit"].map(collapse_unit))
    df["anchor_year_group"] = df["anchor_year_group"].astype("category")
    df["early_vaso"] = df["early_vaso"].astype(int)
    df["early_opioid"] = df["early_opioid"].astype(int)
    df["sud_indication"] = df["sud_indication"].astype(int)
    df["sofa_lite"] = sofa_lite(df)
    df["A"] = df["A"].astype(int)
    df["M"] = df["M"].astype(int)
    df["Y"] = df["vfd_28"].astype(float)
    return df

broad = add_covariates(broad).reset_index(drop=True)
LANDMARK_STAYS = set(lm["stay_id"])
broad["in_landmark"] = broad["stay_id"].isin(LANDMARK_STAYS).astype(int)
analytic = broad[broad["in_landmark"] == 1].reset_index(drop=True).copy()

print(f"\nBROAD target-trial cohort N = {len(broad):,d}  (benzo {broad['A'].mean()*100:.1f}%)")
print(f"LANDMARK mediation cohort N = {len(analytic):,d}  "
      f"({analytic['in_landmark'].size and analytic['M'].mean()*100:.1f}% delirium)")
print(f"  Selected into landmark: {broad['in_landmark'].mean()*100:.1f}% of broad cohort")
print(f"  VFD-28 mean (broad)   : {broad['Y'].mean():.2f}")

## 5.1 - Participant flowchart (STROBE / CONSORT style)

Main-column boxes show the remaining sample after each eligibility gate; side boxes report the number excluded at that step, matching the flowchart convention used in the companion MIMIC studies.


In [ ]:
def draw_flowchart(flow):
    # STROBE/CONSORT-style flow: main stem + side exclusion boxes
    labels = [m[0] for m in flow]
    counts = [m[1] for m in flow]
    excl = [counts[i - 1] - counts[i] for i in range(1, len(counts))]
    n = len(flow)
    fig, ax = plt.subplots(figsize=(10.2, 1.45 * n + 0.6))
    ax.set_xlim(0, 11)
    ax.set_ylim(0, n + 0.6)
    ax.axis("off")

    def y(i):
        return n - i

    for i, (lbl, cnt) in enumerate(zip(labels, counts)):
        fc = PALETTE["panel"] if i in (0, n - 1) else PALETTE["bg"]
        ax.add_patch(FancyBboxPatch(
            (0.9, y(i) - 0.34), 5.0, 0.68,
            boxstyle="round,pad=0.04,rounding_size=0.10",
            fc=fc, ec=PALETTE["ink"], lw=1.2, zorder=3))
        ax.text(3.4, y(i), f"{lbl}\nn = {cnt:,d}", ha="center", va="center",
                color=PALETTE["ink"], fontsize=9.4, fontweight="bold", zorder=4)
        if i < n - 1:
            ax.add_patch(FancyArrowPatch(
                (3.4, y(i) - 0.34), (3.4, y(i + 1) + 0.34),
                arrowstyle="-|>", mutation_scale=16, lw=1.4,
                color=PALETTE["ink"], zorder=2))

    for j, e in enumerate(excl):
        yi = (y(j) + y(j + 1)) / 2
        ax.add_patch(FancyBboxPatch(
            (6.5, yi - 0.26), 4.0, 0.52,
            boxstyle="round,pad=0.03,rounding_size=0.08",
            fc="#ffffff", ec=PALETTE["muted"], lw=1.0, zorder=3))
        ax.text(8.5, yi, f"Excluded: {e:,d}", ha="center", va="center",
                color=PALETTE["ink"], fontsize=9.0, zorder=4)
        ax.add_patch(FancyArrowPatch(
            (3.4 + 2.5, yi), (6.5, yi),
            arrowstyle="-|>", mutation_scale=12, lw=1.1,
            color=PALETTE["muted"], ls="--", zorder=2))

    save_fig(fig, "fig01_flow")
    plt.show()

draw_flowchart(flow)


## 6 - Missing-data assessment, Table 1, and MICE

Before modeling we quantify missingness, drop covariates above the 40% rule-of-thumb threshold, report Table 1 on the *observed* analytic sample (with SMDs, not p-values), then generate multiple completed datasets with chained equations (`IterativeImputer`, posterior sampling). The imputation model is congenial: it includes the confounders plus the exposure, mediator, and outcome.


In [ ]:
CONT_VARS = ["age", "gcs", "map", "hr", "lactate", "creatinine", "weight",
             "bilirubin", "platelets", "comorbidity_count", "sofa_lite"]
BIN_VARS  = ["female", "early_vaso", "early_opioid", "sud_indication"]
CAT_VARS  = ["race_eth", "admission_type", "insurance", "careunit_grp", "anchor_year_group"]
ANALYSIS_VARS = CONT_VARS + BIN_VARS + CAT_VARS

# Missingness / imputation model is fit on the BROAD target-trial cohort (the
# superset used for the total effect); the landmark mediation cohort is a subset.
miss = (broad[ANALYSIS_VARS].isna().mean() * 100).sort_values(ascending=False)
miss_tbl = miss.reset_index()
miss_tbl.columns = ["variable", "pct_missing"]
miss_tbl.to_csv(TABDIR / "missingness.csv", index=False)
print(miss_tbl.to_string(index=False))

keep_cont = [v for v in CONT_VARS if miss.get(v, 0) <= CONFIG["MISSING_DROP_THRESHOLD"] * 100]
keep_bin  = [v for v in BIN_VARS  if miss.get(v, 0) <= CONFIG["MISSING_DROP_THRESHOLD"] * 100]
keep_cat  = [v for v in CAT_VARS  if miss.get(v, 0) <= CONFIG["MISSING_DROP_THRESHOLD"] * 100]
print("Kept confounders:", keep_cont + keep_bin + keep_cat)

miss_frac = broad[keep_cont + keep_bin + keep_cat].isna().any(axis=1).mean()
M_IMPUT = int(np.clip(np.ceil(100 * miss_frac), CONFIG["MIN_IMPUTATIONS"], CONFIG["MAX_IMPUTATIONS"]))
print(f"Incomplete-confounder fraction = {miss_frac*100:.1f}%  -> M_IMPUT = {M_IMPUT}")

PASS_COLS = ["stay_id", "A", "M", "Y", "vfd_28", "dead_28d", "days_to_death",
             "vent_days_28d", "rass_mean_24h", "in_landmark", "los",
             "cam_first_pos_h", "cam_second_pos_h", "cam_n_pos",
             "cam_first_assess_h", "cam_last_assess_h", "cam_n_assess",
             "early_midaz", "early_lora", "early_nonbenzo_sed"]
dat = broad[PASS_COLS + keep_cont + keep_bin + keep_cat].copy()
dat["A"] = dat["A"].astype(int)
dat["M"] = dat["M"].astype(int)
dat["Y"] = dat["Y"].astype(float)

# ---- Table 1 on observed (non-imputed) data ----
def smd_cont(x1, x0):
    x1, x0 = np.asarray(x1, float), np.asarray(x0, float)
    x1, x0 = x1[~np.isnan(x1)], x0[~np.isnan(x0)]
    if len(x1) < 2 or len(x0) < 2:
        return np.nan
    sp = np.sqrt((x1.var(ddof=1) + x0.var(ddof=1)) / 2)
    return (x1.mean() - x0.mean()) / sp if sp > 0 else 0.0

def smd_bin(p1, p0):
    sp = np.sqrt((p1 * (1 - p1) + p0 * (1 - p0)) / 2)
    return (p1 - p0) / sp if sp > 0 else 0.0

# Cache the imputation-input frame (broad cohort) so downstream descriptive and
# modeling steps can be rebuilt without re-running the heavy raw-data passes.
dat.to_csv(TABDIR / "_analytic_cache.csv", index=False)

# ---- Table 1: baseline characteristics of the LANDMARK mediation cohort only
#      (no outcomes), formatted to match the companion MIMIC studies: continuous
#      = median [IQR], categorical = n (%) under an indented header, with an
#      Overall column, a Missing (%) column, and the exposed-vs-unexposed SMD.
t1src = dat[dat["in_landmark"] == 1].copy()
EXP = t1src[t1src["A"] == 1]     # exposed = early benzodiazepine
UNEXP = t1src[t1src["A"] == 0]   # unexposed = non-benzo sedation

def _miss(v):
    return f"{t1src[v].isna().mean() * 100:.1f}"

def _mi(s):
    s = pd.to_numeric(s, errors="coerce").dropna()
    q1, q2, q3 = s.quantile([0.25, 0.50, 0.75])
    return f"{q2:.1f} [{q1:.1f}, {q3:.1f}]"

def _np(mask):
    mask = mask.astype(bool)
    return f"{int(mask.sum()):,d} ({mask.mean() * 100:.1f})"

CONT_LABELS = {
    "age": "Age, years, median [IQR]",
    "gcs": "Glasgow Coma Scale, median [IQR]",
    "map": "Mean arterial pressure, mmHg, median [IQR]",
    "hr": "Heart rate, bpm, median [IQR]",
    "lactate": "Lactate, mmol/L, median [IQR]",
    "creatinine": "Creatinine, mg/dL, median [IQR]",
    "weight": "Weight, kg, median [IQR]",
    "platelets": "Platelets, K/uL, median [IQR]",
    "comorbidity_count": "Comorbidity count, median [IQR]",
    "sofa_lite": "SOFA-lite severity score, median [IQR]",
}
BIN_SPECS = {
    "female": ("Sex", [("Female", 1), ("Male", 0)]),
    "early_vaso": ("Early vasopressor use, first 24 h", [("No", 0), ("Yes", 1)]),
    "early_opioid": ("Early opioid co-sedation, first 24 h", [("No", 0), ("Yes", 1)]),
    "sud_indication": ("Benzodiazepine indication (alcohol/sedative/seizure)", [("No", 0), ("Yes", 1)]),
}
CAT_HEADERS = {
    "race_eth": "Race/ethnicity",
    "admission_type": "Admission type",
    "insurance": "Insurance",
    "careunit_grp": "ICU unit type",
    "anchor_year_group": "Admission era",
}

rows = [("No. of ICU stays", f"{len(t1src):,d}", f"{len(UNEXP):,d}", f"{len(EXP):,d}", "", "")]

for v in keep_cont:
    rows.append((CONT_LABELS.get(v, v), _mi(t1src[v]), _mi(UNEXP[v]), _mi(EXP[v]),
                 _miss(v), f"{smd_cont(EXP[v], UNEXP[v]):+.3f}"))

for v in keep_bin:
    header, levels = BIN_SPECS.get(v, (v, [("No", 0), ("Yes", 1)]))
    rows.append((header, "", "", "", _miss(v), ""))
    for lab, code in levels:
        pe, pu = (EXP[v] == code).mean(), (UNEXP[v] == code).mean()
        rows.append((f"    {lab}", _np(t1src[v] == code), _np(UNEXP[v] == code),
                     _np(EXP[v] == code), "", f"{smd_bin(pe, pu):+.3f}"))

for v in keep_cat:
    rows.append((CAT_HEADERS.get(v, v), "", "", "", _miss(v), ""))
    for lev in pd.Categorical(t1src[v]).categories:
        pe, pu = (EXP[v] == lev).mean(), (UNEXP[v] == lev).mean()
        rows.append((f"    {lev}", _np(t1src[v] == lev), _np(UNEXP[v] == lev),
                     _np(EXP[v] == lev), "", f"{smd_bin(pe, pu):+.3f}"))

table1 = pd.DataFrame(rows, columns=["Characteristic", "Overall", "Non-benzo sedation",
                                     "Benzodiazepine", "Missing", "SMD"])
table1.to_csv(TABDIR / "table1_baseline.csv", index=False)
print("Saved -> tables/table1_baseline.csv")
table1


### 6.1 - Descriptive mediation signal

Baseline imbalance is reported in Table 1 (SMDs). Here we show the raw mediator prevalence by exposure and mean ventilator-free days by the exposure x mediator strata.


In [ ]:
# Descriptive mediation signal only (SMDs live in Table 1; Love plot shows IPTW balance)
fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.6))

ax = axes[0]
mvals = [dat.loc[dat["A"] == a, "M"].mean() * 100 for a in (0, 1)]
bars = ax.bar(["Non-benzo", "Benzodiazepine"], mvals,
              color=[PALETTE["unexposed"], PALETTE["exposed"]], edgecolor=PALETTE["ink"])
for b, v in zip(bars, mvals):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.8, f"{v:.1f}%", ha="center", fontweight="bold")
ax.set_ylabel("Incident delirium, %")
ax.set_title("(a) Mediator by exposure", fontsize=12)

ax = axes[1]
labs, yv, ccs = [], [], []
for a in (0, 1):
    for m in (0, 1):
        sub = dat[(dat["A"] == a) & (dat["M"] == m)]
        labs.append(f"A={a}\nM={m}")
        yv.append(sub["Y"].mean() if len(sub) else 0)
        ccs.append(PALETTE["mediator"] if m == 1 else PALETTE["sky"])
bars = ax.bar(range(4), yv, color=ccs, edgecolor=PALETTE["ink"])
for b, v in zip(bars, yv):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.15, f"{v:.1f}", ha="center", fontsize=9, fontweight="bold")
ax.set_xticks(range(4)); ax.set_xticklabels(labs, fontsize=9)
ax.set_ylabel("Mean VFD-28")
ax.set_title("(b) Outcome by A x M", fontsize=12)

fig.tight_layout(); save_fig(fig, "fig02_descriptive"); plt.show()


## 7 - Multiple Imputation by Chained Equations (MICE)

We generate `M_IMPUT` completed datasets. Continuous covariates are imputed on the original scale; categoricals are integer-coded, imputed, then rounded back to valid levels. Exposure, mediator, and outcome are included as auxiliary variables so the imputation model is congenial with the mediation analysis.


In [ ]:
cat_levels = {v: list(pd.Categorical(dat[v]).categories) for v in keep_cat}

base = pd.DataFrame(index=dat.index)
for v in keep_cont + keep_bin:
    base[v] = dat[v].astype("float64")
for v in keep_cat:
    codes = pd.Categorical(dat[v], categories=cat_levels[v]).codes.astype("float64")
    codes[codes < 0] = np.nan
    base[v] = codes

aux = pd.DataFrame({
    "_A": dat["A"].astype("float64").values,
    "_M": dat["M"].astype("float64").values,
    "_Y": dat["Y"].astype("float64").values,
}, index=dat.index)
X = pd.concat([base, aux], axis=1)
impute_cols = keep_cont + keep_bin + keep_cat

imputations = []
for i in range(M_IMPUT):
    mins = [X[c].min(skipna=True) for c in impute_cols] + [-np.inf] * 3
    maxs = [X[c].max(skipna=True) for c in impute_cols] + [np.inf] * 3
    imp = IterativeImputer(
        estimator=BayesianRidge(), sample_posterior=True, max_iter=15,
        random_state=RANDOM_STATE + i, min_value=mins, max_value=maxs)
    filled = pd.DataFrame(imp.fit_transform(X), columns=X.columns, index=X.index)
    out = dat[["stay_id", "A", "M", "Y", "vfd_28", "dead_28d", "days_to_death",
               "vent_days_28d", "rass_mean_24h", "in_landmark", "los",
               "cam_first_pos_h", "cam_second_pos_h", "cam_n_pos",
               "cam_first_assess_h", "cam_last_assess_h", "cam_n_assess",
               "early_midaz", "early_lora", "early_nonbenzo_sed"]].copy()
    for v in keep_cont + keep_bin:
        out[v] = filled[v].astype("float32")
    for v in keep_cat:
        codes = np.clip(np.rint(filled[v].values), 0, len(cat_levels[v]) - 1).astype(int)
        out[v] = pd.Categorical.from_codes(codes, categories=cat_levels[v])
    imputations.append(out)
    print(f"  imputation {i+1}/{M_IMPUT} complete", end="\r")
print(f"\nGenerated {len(imputations)} completed datasets (each n={len(dat):,d}).")


## 8 - Positivity, stabilized IPTW, and covariate balance

For each imputed dataset we estimate the propensity of early benzodiazepine exposure with logistic regression, form stabilized IPTW, and truncate weights at the 1st/99th percentiles. The propensity model includes every baseline confounder that survived the missingness screen, regardless of baseline SMD: including even well-balanced covariates is the conventional, conservative choice, guarding against residual confounding and slightly improving efficiency. Balance for every covariate is displayed in the Love plot, and any left imbalanced after weighting is additionally carried into the outcome/mediator models (doubly robust). A single panel summarizes the weighting: propensity overlap and the stabilized weight distribution on the left, and the Love plot of covariate balance before vs after weighting on the right.

In [ ]:
from sklearn.linear_model import LogisticRegression

# ---- Propensity model: include every retained baseline confounder ----------
# All confounders that survived the missingness screen enter the propensity
# model, regardless of their baseline SMD. Including even well-balanced
# covariates is the conventional, conservative choice: it guards against
# residual confounding and slightly improves the efficiency of the weighted
# estimator, at the cost of a few extra dimensions. Balance for every covariate
# is still audited in the Love plot, and any left imbalanced after weighting is
# additionally carried into the outcome/mediator models (doubly robust).
ps_cont, ps_bin, ps_cat = list(keep_cont), list(keep_bin), list(keep_cat)
print("PS model covariates (all retained confounders):")
print("  continuous/binary:", ps_cont + ps_bin)
print("  categorical:", ps_cat)

def design_matrix(df):
    parts = [df[ps_cont + ps_bin].astype("float64")] if (ps_cont + ps_bin) else []
    for v in ps_cat:
        d = pd.get_dummies(df[v].astype("object"), prefix=v, drop_first=True, dtype="float64")
        parts.append(d)
    Xd = pd.concat(parts, axis=1)
    Xd = (Xd - Xd.mean()) / Xd.std(ddof=0).replace(0, 1)
    return sm.add_constant(Xd, has_constant="add")

def stabilized_weights(df, trim=(1, 99)):
    """Stabilized IPTW for the treatment (early benzodiazepine)."""
    Xd = design_matrix(df)
    a = df["A"].astype("float64").values
    clf = LogisticRegression(solver="lbfgs", C=1e6, max_iter=2000, random_state=RANDOM_STATE)
    clf.fit(Xd.values, a)
    ps = clf.predict_proba(Xd.values)[:, 1]
    ps = np.clip(ps, 1e-3, 1 - 1e-3)
    pa = a.mean()
    sw = np.where(a == 1, pa / ps, (1 - pa) / (1 - ps))
    lo, hi = np.percentile(sw, list(trim))
    sw = np.clip(sw, lo, hi)
    return ps, sw

def overlap_weights(df):
    """ATO overlap weights (an alternative to IPTW; bounded, no truncation)."""
    Xd = design_matrix(df)
    a = df["A"].astype("float64").values
    clf = LogisticRegression(solver="lbfgs", C=1e6, max_iter=2000, random_state=RANDOM_STATE)
    clf.fit(Xd.values, a)
    ps = np.clip(clf.predict_proba(Xd.values)[:, 1], 1e-3, 1 - 1e-3)
    return np.where(a == 1, 1 - ps, ps)

def selection_weights(dfb):
    """Stabilized inverse-probability-of-selection weight for being retained in
    the landmark mediation cohort, given baseline confounders and treatment.
    Reweights the landmark sample back to the broad early-eligible population."""
    Xd = design_matrix(dfb)
    a = dfb["A"].astype("float64").values.reshape(-1, 1)
    Xmat = np.hstack([Xd.values, a])
    s = dfb["in_landmark"].astype("float64").values
    clf = LogisticRegression(solver="lbfgs", C=1e6, max_iter=2000, random_state=RANDOM_STATE)
    clf.fit(Xmat, s)
    p = np.clip(clf.predict_proba(Xmat)[:, 1], 1e-3, 1 - 1e-3)
    a1 = dfb["A"].values == 1
    num = np.where(a1, s[a1].mean(), s[~a1].mean())   # stabilizing numerator P(S=1|A)
    return num / p

def mediation_frame(dfb, weight="iptw", trim=(1, 99)):
    """Subset a broad imputation to the landmark cohort and attach the mediation
    weight = treatment weight x selection weight (targeting the broad population)."""
    if weight == "none":
        wA = np.ones(len(dfb)); wsel = np.ones(len(dfb))
    elif weight == "overlap":
        wA = overlap_weights(dfb); wsel = selection_weights(dfb)
    else:
        _, wA = stabilized_weights(dfb, trim=trim); wsel = selection_weights(dfb)
    d = dfb.copy()
    d["_w"] = wA * wsel
    d = d[d["in_landmark"] == 1].copy()
    lo, hi = np.percentile(d["_w"], list(trim))
    d["sw"] = np.clip(d["_w"], lo, hi)
    return d.reset_index(drop=True)

# Treatment weights on the BROAD cohort (used for the total effect) ...
for i, df in enumerate(imputations):
    ps, sw = stabilized_weights(df)
    df["ps"], df["sw"] = ps, sw
    df["sw_sel"] = selection_weights(df)

# ... and the landmark mediation frames (IPTW x selection weight).
land_imputations = [mediation_frame(df) for df in imputations]

b0, l0 = imputations[0], land_imputations[0]
print(f"Broad IPTW (imp 1):        mean={b0['sw'].mean():.3f} "
      f"min={b0['sw'].min():.3f} max={b0['sw'].max():.3f}")
print(f"Landmark mediation weight: mean={l0['sw'].mean():.3f} "
      f"min={l0['sw'].min():.3f} max={l0['sw'].max():.3f}  n={len(l0):,d}")
l0[["A", "sw"]].to_csv(TABDIR / "_ps_cache.csv", index=False)


### 8.1 - Covariate balance (Love plot)

Successful weighting pulls absolute SMDs left of the 0.1 guideline (panel c above). This is the primary evidence that measured confounding of the exposure has been controlled; residual imbalance (|SMD| > 0.1 after weighting) is carried into the outcome/mediator models as doubly robust adjusters.

In [ ]:
def weighted_mean_var(x, w):
    w = np.asarray(w, float); x = np.asarray(x, float)
    w = w / w.sum()
    m = np.sum(w * x)
    v = np.sum(w * (x - m) ** 2)
    return m, v

def balance_table(df):
    rows = []
    a = df["A"].values
    w = df["sw"].values
    for v in keep_cont + keep_bin:
        x = df[v].astype("float64").values
        m1, v1 = df.loc[a == 1, v].mean(), df.loc[a == 1, v].var(ddof=1)
        m0, v0 = df.loc[a == 0, v].mean(), df.loc[a == 0, v].var(ddof=1)
        s = np.sqrt((v1 + v0) / 2) or 1.0
        pre = abs(m1 - m0) / s
        wm1, wv1 = weighted_mean_var(x[a == 1], w[a == 1])
        wm0, wv0 = weighted_mean_var(x[a == 0], w[a == 0])
        sw_ = np.sqrt((wv1 + wv0) / 2) or 1.0
        post = abs(wm1 - wm0) / sw_
        rows.append((v, pre, post))
    for v in keep_cat:
        for lvl in cat_levels[v]:
            ind = (df[v].astype(str) == str(lvl)).astype(float).values
            p1, p0 = ind[a == 1].mean(), ind[a == 0].mean()
            s = np.sqrt((p1 * (1 - p1) + p0 * (1 - p0)) / 2) or 1.0
            pre = abs(p1 - p0) / s
            wp1, _ = weighted_mean_var(ind[a == 1], w[a == 1])
            wp0, _ = weighted_mean_var(ind[a == 0], w[a == 0])
            sw_ = np.sqrt((wp1 * (1 - wp1) + wp0 * (1 - wp0)) / 2) or 1.0
            post = abs(wp1 - wp0) / sw_
            rows.append((f"{v}:{lvl}", pre, post))
    return pd.DataFrame(rows, columns=["cov", "pre", "post"])

# Primary balance evidence is the TREATMENT model in the broad cohort under
# stabilized IPTW (what the weights are designed to balance). The Love plot shows
# this. We additionally record balance under the mediation weight (IPTW x
# selection) - noisier because it also reweights for landmark selection - and
# under overlap weights (which achieve exact mean balance by construction).
bal = balance_table(imputations[0]).sort_values("pre")
bal.to_csv(TABDIR / "table_love_balance.csv", index=False)

_med0 = land_imputations[0].copy()
balance_table(_med0).to_csv(TABDIR / "table_love_balance_mediation.csv", index=False)
_ov0 = imputations[0].copy(); _ov0["sw"] = overlap_weights(_ov0)
balance_table(_ov0).to_csv(TABDIR / "table_love_balance_overlap.csv", index=False)
print(f"Max post-|SMD|: IPTW(broad)={bal['post'].max():.3f}  "
      f"mediation-wt={balance_table(_med0)['post'].max():.3f}  "
      f"overlap={balance_table(_ov0)['post'].max():.3f}")

# Combined positivity + balance panel: full-height Love plot on the left,
# propensity overlap and weight distribution stacked on the right.
ps0 = imputations[0]
wt0 = imputations[0]
fig = plt.figure(figsize=(15.5, 9.6))
gs = fig.add_gridspec(2, 2, width_ratios=[1.05, 1.0], height_ratios=[1, 1],
                      left=0.055, right=0.985, top=0.945, bottom=0.075,
                      wspace=0.16, hspace=0.30)
ax_love = fig.add_subplot(gs[:, 0])
ax_ps = fig.add_subplot(gs[0, 1])
ax_w = fig.add_subplot(gs[1, 1])

# (a) propensity overlap
for a, lbl, fc in [(0, "Non-benzo", PALETTE["unexposed"]), (1, "Benzodiazepine", PALETTE["exposed"])]:
    ax_ps.hist(ps0.loc[ps0["A"] == a, "ps"], bins=35, alpha=0.75, color=fc, label=lbl,
               edgecolor=PALETTE["ink"], linewidth=0.4, density=True)
ax_ps.set_xlabel("Estimated propensity score"); ax_ps.set_ylabel("Density")
ax_ps.legend(frameon=False)
ax_ps.set_title("(b) Positivity: propensity overlap", fontsize=13, loc="left")

# (b) stabilized IPTW distribution (broad cohort)
ax_w.hist(wt0["sw"], bins=45, color=PALETTE["direct"], edgecolor="white", linewidth=0.4)
ax_w.axvline(wt0["sw"].mean(), color=PALETTE["ink"], lw=1.5, ls="--",
             label=f"mean = {wt0['sw'].mean():.2f}")
ax_w.set_xlabel("Stabilized IPTW (broad cohort)"); ax_w.set_ylabel("ICU stays")
ax_w.legend(frameon=False)
ax_w.set_title("(c) Weight distribution (1st/99th truncated)", fontsize=13, loc="left")

# (c) Love plot, full height
y = np.arange(len(bal))
ax_love.hlines(y, bal["post"], bal["pre"], color=PALETTE["grid"], lw=1.6, zorder=1)
ax_love.scatter(bal["pre"], y, s=54, facecolors="white", edgecolors=PALETTE["muted"],
                linewidths=1.5, label="Unweighted", zorder=3)
ax_love.scatter(bal["post"], y, s=54, color=PALETTE["direct"], label="IPTW-weighted", zorder=4)
ax_love.axvline(0.1, color=PALETTE["exposed"], ls="--", lw=1.5)
ax_love.set_yticks(y); ax_love.set_yticklabels(bal["cov"], fontsize=8.5)
ax_love.set_ylim(-0.6, len(bal) - 0.4)
ax_love.set_xlabel("Absolute standardized mean difference")
ax_love.set_title("(a) Love plot: covariate balance before vs after IPTW", fontsize=13, loc="left")
ax_love.legend(loc="lower right", frameon=False)
save_fig(fig, "fig03_positivity_balance"); plt.show()

BALANCE_THRESHOLD = 0.1
dr_cont = [v for v in keep_cont + keep_bin
           if (bal.loc[bal["cov"] == v, "post"].iloc[0] > BALANCE_THRESHOLD
               if (bal["cov"] == v).any() else False)]
dr_cat = []
for v in keep_cat:
    sub = bal[bal["cov"].str.startswith(v + ":")]
    if len(sub) and sub["post"].max() > BALANCE_THRESHOLD:
        dr_cat.append(v)
print("Residual post-IPTW imbalance (|SMD| > 0.1) carried as DR adjusters:")
print("  continuous/binary:", dr_cont)
print("  categorical:", dr_cat)


## 9 - IPTW-weighted causal mediation (continuous VFD)

Within each imputed, IPTW-weighted pseudo-population we fit:

- **Mediator model** (weighted logistic): logit P(M = 1 | A, C\*)  
- **Outcome model** (weighted linear): E[Y | A, M, C\*] = theta0 + theta1 A + theta2 M + theta3 (A x M) + ...

where C\* is the residual doubly robust adjustment set (covariates still imbalanced after weighting; if none, the models are IPTW-only). Natural effects are obtained by g-computation of Q(a, a'). Inference uses Rubin's rules across the multiply imputed datasets: within each imputation a nonparametric bootstrap (re-estimating the propensity score, weights, and both models on every draw) gives the within-imputation variance, and these are combined with the between-imputation variance so the reported 95% CIs reflect both sampling and imputation uncertainty.


In [ ]:
# ---- Adjustment sets ------------------------------------------------------
# Mediation identification needs symmetric control of mediator-outcome (M-Y)
# confounding, not only exposure (A) confounding. We therefore adjust BOTH the
# mediator and outcome models for the FULL baseline confounder set C (measured
# at/around time zero, before the landmark and the mediator), in addition to the
# IPTW-x-selection weighting. This is stronger than the previous residual-only
# doubly robust set.
def full_formula():
    terms = list(keep_cont) + list(keep_bin) + [f"C({v})" for v in keep_cat]
    return " + ".join(terms) if terms else "1"

CTRL_FIT = full_formula()
MED_F = f"M ~ A + {CTRL_FIT}"
OUT_F = f"Y ~ A * M + {CTRL_FIT}"
print("Mediator/outcome adjustment set: full baseline confounder set C "
      f"({len(keep_cont) + len(keep_bin) + len(keep_cat)} variables)")

def fit_pair_weighted(d):
    w = d["sw"].astype(float).values
    w = w / w.mean()
    m = smf.glm(MED_F, data=d, family=sm.families.Binomial(), var_weights=w).fit()
    y = smf.wls(OUT_F, data=d, weights=w).fit()
    return m, y

def gcomp_weighted(d, m, y):
    """Natural-effect decomposition (Valeri-VanderWeele) by weighted g-computation."""
    W = d["sw"].astype(float).values
    W = W / W.sum()

    def pM(a):
        dd = d.copy(); dd["A"] = a
        return m.predict(dd).values

    def pY(a, mm):
        dd = d.copy(); dd["A"] = a; dd["M"] = mm
        return y.predict(dd).values

    def Q(a, ap):
        pm = pM(ap)
        return pY(a, 1) * pm + pY(a, 0) * (1 - pm)

    q00, q10, q11 = np.sum(W * Q(0, 0)), np.sum(W * Q(1, 0)), np.sum(W * Q(1, 1))
    te, nde, nie = q11 - q00, q10 - q00, q11 - q10
    return {"Q00": q00, "Q10": q10, "Q11": q11, "TE": te, "NDE": nde, "NIE": nie,
            "PM": (nie / te) if abs(te) > 1e-9 else np.nan}

def interventional_effects(d, m, y):
    """Interventional (in)direct effects: draw M from its population-standardized
    distribution under a' rather than a subject's cross-world value. More
    defensible than natural effects when cross-world assumptions are shaky."""
    W = d["sw"].astype(float).values; W = W / W.sum()

    def gbar(ap):                       # weighted-average P(M=1 | A=a', C)
        dd = d.copy(); dd["A"] = ap
        return float(np.sum(W * m.predict(dd).values))

    def Yint(a, ap):
        g = gbar(ap)
        dd1 = d.copy(); dd1["A"] = a; dd1["M"] = 1
        dd0 = d.copy(); dd0["A"] = a; dd0["M"] = 0
        return float(np.sum(W * (y.predict(dd1).values * g + y.predict(dd0).values * (1 - g))))

    y00, y10, y11 = Yint(0, 0), Yint(1, 0), Yint(1, 1)
    return {"TE_i": y11 - y00, "IDE": y10 - y00, "IIE": y11 - y10}

def total_effect_broad(dfb):
    """IPTW-standardized total effect on VFD in the broad cohort (Y ~ A + C)."""
    w = dfb["sw"].astype(float).values; w = w / w.mean()
    ym = smf.wls(f"Y ~ A + {CTRL_FIT}", data=dfb, weights=w).fit()
    W = w / w.sum()
    y1 = float(np.sum(W * ym.predict(dfb.assign(A=1)).values))
    y0 = float(np.sum(W * ym.predict(dfb.assign(A=0)).values))
    return y1 - y0

def mediation_from_broad(dfb, weight="iptw", trim=(1, 99)):
    """End-to-end mediation estimate from a broad imputation/resample: recompute
    treatment + selection weights, subset to landmark, fit and decompose."""
    d = mediation_frame(dfb, weight=weight, trim=trim)
    m, y = fit_pair_weighted(d)
    out = gcomp_weighted(d, m, y)
    out.update(interventional_effects(d, m, y))
    return out

# ---- Point estimates + full multiple-imputation inference (Rubin's rules) ----
# Per imputation: point estimate on the landmark frame, plus a nonparametric
# bootstrap that resamples the BROAD cohort and re-estimates treatment weights,
# selection weights, landmark subsetting, and both models on every draw. The
# per-imputation bootstrap variance (within) is combined with the between-
# imputation variance by Rubin's rules.
keys = ["TE", "NDE", "NIE", "PM", "TE_i", "IDE", "IIE"]
BOOT_PER_IMP = max(40, CONFIG["N_BOOT"] // 3)

point_list, U_list, te_broad_list, te_broad_U = [], [], [], []
for i, dfb in enumerate(imputations):
    d = land_imputations[i]
    m, y = fit_pair_weighted(d)
    qi = gcomp_weighted(d, m, y); qi.update(interventional_effects(d, m, y))
    point_list.append(qi)
    te_broad_list.append(total_effect_broad(dfb))

    bvals = {k: [] for k in keys}; teb = []
    brng = np.random.default_rng(RANDOM_STATE + 1000 * i)
    ix = np.arange(len(dfb))
    for _ in range(BOOT_PER_IMP):
        db = dfb.iloc[brng.choice(ix, size=len(dfb), replace=True)].reset_index(drop=True)
        try:
            ps, sw = stabilized_weights(db); db["ps"], db["sw"] = ps, sw
            db["sw_sel"] = selection_weights(db)
            rb = mediation_from_broad(db)
            for k in keys:
                bvals[k].append(rb[k])
            teb.append(total_effect_broad(db))
        except Exception:
            continue
    U_list.append({k: np.nanvar(bvals[k], ddof=1) if len(bvals[k]) > 1 else np.nan for k in keys})
    te_broad_U.append(np.nanvar(teb, ddof=1) if len(teb) > 1 else np.nan)
    print(f"  imput {i+1}: NDE={qi['NDE']:+.2f} NIE={qi['NIE']:+.2f} "
          f"TE(broad)={te_broad_list[-1]:+.2f}  (boot {len(bvals['NIE'])}/{BOOT_PER_IMP})")

M = len(imputations)
def rubin(points, Us):
    q = np.asarray(points, float)
    Qbar = np.nanmean(q)
    Bvar = np.nanvar(q, ddof=1) if len(q) > 1 else 0.0
    Ubar = np.nanmean(Us)
    T = Ubar + (1.0 + 1.0 / len(q)) * Bvar
    se = math.sqrt(T) if T > 0 else 0.0
    return float(Qbar), (Qbar - 1.96 * se, Qbar + 1.96 * se)

est, ci = {}, {}
for k in ["Q00", "Q10", "Q11"]:
    est[k] = float(np.mean([p[k] for p in point_list]))
for k in keys:
    est[k], ci[k] = rubin([p[k] for p in point_list], [u[k] for u in U_list])
est["TE_broad"], ci["TE_broad"] = rubin(te_broad_list, te_broad_U)
print("Pooled estimates:", {k: round(est[k], 3) for k in ["TE", "NDE", "NIE", "PM", "TE_broad"]})
print("Interventional:", {k: round(est[k], 3) for k in ["TE_i", "IDE", "IIE"]})
print("Rubin 95% CIs:", {k: (round(ci[k][0], 3), round(ci[k][1], 3)) for k in ["NDE", "NIE", "TE_broad"]})

# ---- Second estimator: inverse-odds-ratio mediator weighting (IORW) --------
# A distinct identification strategy that handles the mediator with WEIGHTS
# rather than an outcome model (Tchetgen Tchetgen 2013). Exposed units are
# weighted by the inverse odds ratio f(M|A=0,C)/f(M|A=1,C) to break the A->M
# path; the treatment coefficient in the IPTW-weighted outcome regression then
# estimates the direct effect, and NIE = TE - NDE. Agreement with g-computation
# is reassuring and directly answers the mediator-IPW request.
def iorw_on_frame(d):
    w = d["sw"].astype(float).values; w = w / w.mean()
    mmod = smf.glm(MED_F, data=d, family=sm.families.Binomial(), var_weights=w).fit()
    pm1 = mmod.predict(d.assign(A=1)).values
    pm0 = mmod.predict(d.assign(A=0)).values
    Mo = d["M"].astype(float).values
    fa1 = np.clip(np.where(Mo == 1, pm1, 1 - pm1), 1e-6, None)
    fa0 = np.where(Mo == 1, pm0, 1 - pm0)
    ior = np.where(d["A"].values == 1, fa0 / fa1, 1.0)
    te = smf.wls(f"Y ~ A + {CTRL_FIT}", data=d, weights=w).fit().params["A"]
    nde = smf.wls(f"Y ~ A + {CTRL_FIT}", data=d, weights=w * ior).fit().params["A"]
    return te, nde, te - nde

def _pooled_with_boot(point_fn, seed, keys):
    pts = np.array([point_fn(d) for d in land_imputations], float)
    pt = pts.mean(axis=0)
    rng = np.random.default_rng(seed); dfb0 = imputations[0]; ix = np.arange(len(dfb0))
    bs = []
    for _ in range(max(80, CONFIG["N_BOOT"] // 3)):
        db = dfb0.iloc[rng.choice(ix, size=len(dfb0), replace=True)].reset_index(drop=True)
        try:
            ps, sw = stabilized_weights(db); db["ps"], db["sw"] = ps, sw
            db["sw_sel"] = selection_weights(db)
            bs.append(point_fn(mediation_frame(db, weight="iptw")))
        except Exception:
            pass
    bs = np.array(bs, float)
    out = {}
    for j, k in enumerate(keys):
        lo, hi = (np.nanpercentile(bs[:, j], [2.5, 97.5]) if len(bs) > 5 else (np.nan, np.nan))
        out[k] = (float(pt[j]), (float(lo), float(hi)))
    return out

iorw = _pooled_with_boot(iorw_on_frame, RANDOM_STATE + 500, ["TE_ipw", "NDE_ipw", "NIE_ipw"])
for k, (p, c) in iorw.items():
    est[k], ci[k] = p, c

# ---- Overlap-weighted mediation (co-primary weighting scheme) --------------
# Overlap (ATO) weights achieve exact mean balance on every PS covariate by
# construction, so they are reported co-primary alongside IPTW x selection.
def _ovl_point(d):
    dd = d.copy(); dd["sw"] = overlap_weights(dd)
    r = gcomp_weighted(dd, *fit_pair_weighted(dd))
    return r["TE"], r["NDE"], r["NIE"]
ovl = _pooled_with_boot(_ovl_point, RANDOM_STATE + 600, ["TE_ovl", "NDE_ovl", "NIE_ovl"])
for k, (p, c) in ovl.items():
    est[k], ci[k] = p, c
print(f"IORW: NDE={est['NDE_ipw']:+.2f} NIE={est['NIE_ipw']:+.2f} | "
      f"Overlap: NDE={est['NDE_ovl']:+.2f} NIE={est['NIE_ovl']:+.2f}")


## 10 - Primary results panel: E-value, mediation, and sensitivity forest

One publication figure: **(a)** E-value contour (large, left); **(b)** IPTW-weighted mediation decomposition with proportion mediated (waterfall + donut as a single wide panel, top right); **(c)** robustness forest for the natural indirect effect (bottom right).


In [ ]:
def fmt(k):
    return f"{est[k]:+.2f} ({ci[k][0]:+.2f}, {ci[k][1]:+.2f})"

table2 = pd.DataFrame([
    ("Total effect, broad cohort (VFD days)", fmt("TE_broad")),
    ("Total effect, landmark cohort (VFD days)", fmt("TE")),
    ("Natural direct effect (NDE), VFD days", fmt("NDE")),
    ("Natural indirect effect (NIE), VFD days", fmt("NIE")),
    ("Proportion mediated", f"{est['PM']*100:.1f}% ({ci['PM'][0]*100:.1f}%, {ci['PM'][1]*100:.1f}%)"),
    ("Interventional direct effect (IDE), VFD days", fmt("IDE")),
    ("Interventional indirect effect (IIE), VFD days", fmt("IIE")),
    ("NDE - mediator IPW (IORW), VFD days", fmt("NDE_ipw")),
    ("NIE - mediator IPW (IORW), VFD days", fmt("NIE_ipw")),
    ("NDE - overlap weights, VFD days", fmt("NDE_ovl")),
    ("NIE - overlap weights, VFD days", fmt("NIE_ovl")),
], columns=["Effect", "Estimate (95% CI)"])
table2.to_csv(TABDIR / "table2_mediation.csv", index=False)
print("Counterfactual mean VFD (standardized): "
      f"Q00={est['Q00']:.2f}  Q10={est['Q10']:.2f}  Q11={est['Q11']:.2f}")
print(table2.to_string(index=False))

# ---- Sensitivity battery for the NIE ---------------------------------------
# Each spec transforms the BROAD imputations, then runs the full mediation
# pipeline (treatment + selection weights, landmark subset, g-computation).
# Points are pooled across all imputations; the 95% CI is a nonparametric
# bootstrap on imputation 1 (resampling the broad cohort each draw).
def spec_midaz(dfb):
    d = dfb.copy()
    d["A"] = np.where(d["early_midaz"] == 1, 1,
              np.where((d["early_nonbenzo_sed"] == 1) & (d["early_midaz"] == 0) & (d["early_lora"] == 0), 0, np.nan))
    return d.dropna(subset=["A"]).assign(A=lambda x: x["A"].astype(int)).reset_index(drop=True)

def spec_nodeep(dfb):
    return dfb[dfb["rass_mean_24h"].isna() | (dfb["rass_mean_24h"] > -4)].reset_index(drop=True)

def spec_no_indication(dfb):
    return dfb[dfb["sud_indication"] == 0].reset_index(drop=True)

def spec_no_cvicu(dfb):
    return dfb[dfb["careunit_grp"].astype(str) != "CVICU"].reset_index(drop=True)

SENS_SPECS = [
    ("Midazolam-only",    spec_midaz,         {"weight": "iptw"}),
    ("No deep sedation",  spec_nodeep,        {"weight": "iptw"}),
    ("Exclude benzo indication", spec_no_indication, {"weight": "iptw"}),
    ("Exclude CVICU",     spec_no_cvicu,      {"weight": "iptw"}),
    ("Overlap weights",   (lambda d: d),      {"weight": "overlap"}),
    ("Unweighted",        (lambda d: d),      {"weight": "none"}),
]

def _nie_point(spec_fn, kw):
    vals = []
    for dfb in imputations:
        try:
            vals.append(mediation_from_broad(spec_fn(dfb), **kw)["NIE"])
        except Exception:
            pass
    return float(np.nanmean(vals)) if vals else np.nan

def _nie_ci(spec_fn, kw, seed, nb=None):
    nb = nb or max(80, CONFIG["N_BOOT"] // 3)
    dfb0 = imputations[0]; ix = np.arange(len(dfb0)); rng = np.random.default_rng(seed)
    bs = []
    for _ in range(nb):
        db = spec_fn(dfb0.iloc[rng.choice(ix, size=len(dfb0), replace=True)].reset_index(drop=True))
        try:
            bs.append(mediation_from_broad(db, **kw)["NIE"])
        except Exception:
            pass
    return (np.nanpercentile(bs, 2.5), np.nanpercentile(bs, 97.5)) if len(bs) > 5 else (np.nan, np.nan)

sens = {}
for j, (name, fn, kw) in enumerate(SENS_SPECS):
    pt = _nie_point(fn, kw)
    lo, hi = _nie_ci(fn, kw, RANDOM_STATE + 100 + j)
    sens[name] = (pt, (lo, hi))
    print(f"  {name:26s} NIE={pt:+.2f}  CI=({lo:+.2f}, {hi:+.2f})")

# ---- E-values --------------------------------------------------------------
def e_value(rr):
    rr = rr if rr >= 1 else 1 / max(rr, 1e-9)
    return rr + math.sqrt(rr * (rr - 1))

# (a) Total effect on the low-VFD risk scale, estimated in the BROAD cohort.
d0 = imputations[0].copy()
med_y = d0["Y"].median()
d0["low_vfd"] = (d0["Y"] <= med_y).astype(int)

def low_vfd_rr(df):
    w = df["sw"] / df["sw"].mean()
    mod = smf.glm(f"low_vfd ~ A + {CTRL_FIT}", data=df, family=sm.families.Binomial(),
                  var_weights=w).fit()
    r1 = mod.predict(df.assign(A=1)).mean()
    r0 = mod.predict(df.assign(A=0)).mean()
    return r1, r0, r1 / max(r0, 1e-9)

r1, r0, rr_te = low_vfd_rr(d0)
ev = e_value(rr_te)
_brng = np.random.default_rng(RANDOM_STATE + 7); _ix = np.arange(len(d0)); rr_boot = []
for _ in range(CONFIG["N_BOOT"]):
    db = d0.iloc[_brng.choice(_ix, size=len(d0), replace=True)].reset_index(drop=True)
    try:
        ps, sw = stabilized_weights(db); db["sw"] = sw
        rr_boot.append(low_vfd_rr(db)[2])
    except Exception:
        continue
rr_lo, rr_hi = np.nanpercentile(rr_boot, [2.5, 97.5])
rr_ci_near_null = rr_lo if rr_te >= 1 else rr_hi
ev_ci = e_value(rr_ci_near_null)

# (b) E-value for the NIE via the standardized-mean-difference approximation
# (VanderWeele & Ding 2017: RR ~= exp(0.91 x d) for a standardized difference d).
sd_y = float(land_imputations[0]["Y"].std(ddof=1))
def rr_from_diff(diff):
    return math.exp(0.91 * abs(diff) / sd_y) if sd_y > 0 else 1.0
rr_nie = rr_from_diff(est["NIE"])
ev_nie = e_value(rr_nie)
nie_near_null = ci["NIE"][1] if est["NIE"] < 0 else ci["NIE"][0]
ev_nie_ci = e_value(rr_from_diff(nie_near_null)) if (est["NIE"] * nie_near_null > 0) else 1.0
print(f"Low-VFD IPTW risks (broad): R1={r1:.3f} R0={r0:.3f} RR={rr_te:.2f} ({rr_lo:.2f}, {rr_hi:.2f})")
print(f"E-value TE: point={ev:.2f}  CI limit={ev_ci:.2f}")
print(f"E-value NIE: point={ev_nie:.2f}  CI limit={ev_nie_ci:.2f}  (approx RR={rr_nie:.2f})")

# ===================== COMBINED PANEL fig04 =====================
# Clean-sheet layout on an 18 x 9 canvas.
#   (a) E-value contour, full height, colorbar attached at its right.
#   (b) mediation bars + large donut, (c) forest: one shared spine box,
#       one shared right edge, one shared top line with (a).
# All titles sit on the same baseline. Save WITHOUT bbox='tight'.
import matplotlib.transforms as mtransforms

FIGW, FIGH = 18.0, 9.0
fig = plt.figure(figsize=(FIGW, FIGH))

MARG_L, MARG_R, MARG_T, MARG_B = 0.052, 0.982, 0.905, 0.095
ROW_GAP = 0.120

# Left block: square contour, height = full working height
work_h = MARG_T - MARG_B
axe_w = work_h * (FIGH / FIGW)            # square in inches
axe_x = MARG_L
CB_PAD, CB_W = 0.006, 0.013
cax_x = axe_x + axe_w + CB_PAD

# Right block starts after a real gutter; ends flush at MARG_R
right_x = cax_x + CB_W + 0.150            # gutter holds (c) y-labels
right_w = MARG_R - right_x
panel_h = (work_h - ROW_GAP) / 2.0
top_y = MARG_B + panel_h + ROW_GAP

axe = fig.add_axes([axe_x, MARG_B, axe_w, work_h])
cax = fig.add_axes([cax_x, MARG_B + 0.10 * work_h, CB_W, 0.80 * work_h])

# (b): bars left, donut right, both inside the same band as (c)
BAR_FRAC = 0.60
bars_w = right_w * BAR_FRAC
axw = fig.add_axes([right_x, top_y, bars_w, panel_h])
# Donut: largest square that fits the remaining slot, flush right
slot_x = right_x + bars_w
slot_w = right_w - bars_w
donut_side_h = 0.80 * panel_h
donut_w_eq = donut_side_h * (FIGH / FIGW)
donut_w_eq = min(donut_w_eq, slot_w * 0.92)
donut_side_h = donut_w_eq * (FIGW / FIGH)
axd = fig.add_axes([
    slot_x + (slot_w - donut_w_eq) / 2.0,
    top_y + 0.52 * (panel_h - donut_side_h),
    donut_w_eq, donut_side_h,
])
axf = fig.add_axes([right_x, MARG_B, right_w, panel_h])

# ---------- (a) E-value ----------
bias_needed = rr_te if rr_te >= 1 else 1 / rr_te
xmax = max(6.0, ev + 1.5)
xx = np.linspace(1.0, xmax, 400)
A_, Bmesh = np.meshgrid(xx, xx)
BF = (A_ * Bmesh) / (A_ + Bmesh - 1)
cs = axe.contourf(A_, Bmesh, BF, levels=18, cmap=CMAP_SEQ, alpha=0.92)
target = axe.contour(A_, Bmesh, BF, levels=[bias_needed],
                     colors=[PALETTE["warn"]], linewidths=2.4)
axe.clabel(target, fmt={bias_needed: f"explains RR = {bias_needed:.2f}"},
           fontsize=10.5, inline_spacing=6)
axe.scatter([ev], [ev], s=170, color=PALETTE["total"],
            edgecolor="white", lw=1.4, zorder=5)
axe.annotate(
    f"E-value = {ev:.2f}", (ev, ev), xytext=(ev + 0.75, ev - 1.05),
    fontsize=14, fontweight="bold", color=PALETTE["total"],
    arrowprops=dict(arrowstyle="-|>", color=PALETTE["total"], lw=1.5,
                    shrinkA=2, shrinkB=6),
)
axe.set_xlabel("RR: confounder -> exposure", fontsize=12.5)
axe.set_ylabel("RR: confounder -> outcome", fontsize=12.5)
axe.tick_params(labelsize=11)
axe.set_xlim(1.0, xmax); axe.set_ylim(1.0, xmax)
axe.set_title("(a) E-value for total effect (low VFD)",
              fontsize=15, fontweight="bold", pad=10, loc="left")
cb = fig.colorbar(cs, cax=cax)
cb.set_label("bias factor", fontsize=11)
cb.ax.tick_params(labelsize=9.5)
cb.outline.set_linewidth(0.8)

# ---------- (b) mediation bars ----------
te, nde, nie = est["TE"], est["NDE"], est["NIE"]
te_ci, nde_ci, nie_ci = ci["TE"], ci["NDE"], ci["NIE"]
bar_labels = ["Direct\n(A->Y)", "Indirect\n(A->M->Y)", "Total"]
bar_vals = [nde, nie, te]
bar_cis = [nde_ci, nie_ci, te_ci]
bar_cols = [PALETTE["direct"], PALETTE["indirect"], PALETTE["total"]]
for xi, (v, (lo, hi), col) in enumerate(zip(bar_vals, bar_cis, bar_cols)):
    axw.bar(xi, v, color=col, edgecolor=PALETTE["ink"], width=0.66, zorder=3)
    axw.errorbar(xi, v, yerr=[[v - lo], [hi - v]], color=PALETTE["ink"],
                 capsize=6, lw=1.5, zorder=4)
    axw.text(xi, min(v, lo) - 0.16, f"{v:+.2f}", ha="center", va="top",
             fontweight="bold", color=col, fontsize=13.5)
axw.axhline(0, color=PALETTE["ink"], lw=1.1)
axw.set_xticks([0, 1, 2]); axw.set_xticklabels(bar_labels, fontsize=12)
axw.set_ylabel("Difference in ventilator-free days", fontsize=12.5)
axw.tick_params(axis="y", labelsize=11)
axw.set_xlim(-0.60, 2.60)
lo_all = min(lo for lo, _ in bar_cis)
axw.set_ylim(lo_all - 0.85, 0.45)
axw.set_title("(b) IPTW-weighted mediation decomposition",
              fontsize=15, fontweight="bold", pad=10, loc="left")

# ---------- donut ----------
pm = float(est["PM"]) if np.isfinite(est["PM"]) else 0.0
pm_disp = max(0.0, min(1.0, pm))
axd.pie(
    [max(pm_disp, 1e-6), max(1 - pm_disp, 1e-6)],
    colors=[PALETTE["indirect"], PALETTE["grid"]],
    startangle=90, counterclock=False,
    wedgeprops=dict(width=0.34, edgecolor="white", linewidth=2.5),
)
axd.set_aspect("equal")
axd.text(0, 0.10, f"{est['PM']*100:.0f}%", ha="center", va="center",
         fontsize=36, fontweight="bold", color=PALETTE["indirect"])
axd.text(0, -0.42, "of the total effect", ha="center", va="center",
         fontsize=11, color=PALETTE["muted"])
axd.text(0.5, -0.06, "mediated by incident delirium",
         transform=axd.transAxes, ha="center", va="top",
         fontsize=12.5, color=PALETTE["ink"], clip_on=False)

# ---------- (c) forest ----------
FOREST_SHORT = {
    "Midazolam-only": "Midazolam-only",
    "No deep sedation": "No deep sed.",
    "Exclude benzo indication": "Excl. indication",
    "Exclude CVICU": "Excl. CVICU",
    "Overlap weights": "Overlap wt.",
    "Unweighted": "Unweighted",
}
forest = [("Primary", est["NIE"], ci["NIE"][0], ci["NIE"][1])]
for k, (e, c) in sens.items():
    forest.append((FOREST_SHORT.get(k, k), e, c[0], c[1]))
yy = np.arange(len(forest))[::-1]
f_cols = [PALETTE["total"], PALETTE["direct"], PALETTE["warn"], PALETTE["muted"]]
f_los = [lo if np.isfinite(lo) else v for (_, v, lo, hi) in forest]
f_his = [hi if np.isfinite(hi) else v for (_, v, lo, hi) in forest]
# Data occupy the left ~70% of the axis; the right 30% is a text column.
span = max(f_his) - min(f_los)
xlo = min(f_los) - 0.06 * span
data_hi = max(f_his) + 0.06 * span
xhi = xlo + (data_hi - xlo) / 0.70
axf.set_xlim(xlo, xhi)
trans_text = mtransforms.blended_transform_factory(axf.transAxes, axf.transData)
for i, (lab, v, lo, hi) in enumerate(forest):
    col = f_cols[i % len(f_cols)]
    if np.isfinite(lo) and np.isfinite(hi):
        axf.plot([lo, hi], [yy[i], yy[i]], color=col, lw=3.0,
                 solid_capstyle="round", zorder=2)
        txt = f"{v:+.2f} ({lo:+.2f}, {hi:+.2f})"
    else:
        txt = f"{v:+.2f} (point estimate)"
    axf.scatter([v], [yy[i]], s=160, color=col, edgecolor=PALETTE["ink"],
                lw=1.0, zorder=3)
    axf.text(0.985, yy[i], txt, transform=trans_text,
             va="center", ha="right", fontsize=12, color=PALETTE["ink"])
axf.axvline(0, color=PALETTE["ink"], ls="--", lw=1.2, zorder=1)
axf.set_yticks(yy)
axf.set_yticklabels([f[0] for f in forest], fontsize=11.5)
axf.tick_params(axis="y", pad=8, length=0)
axf.tick_params(axis="x", labelsize=11)
axf.set_xlabel("Natural indirect effect (VFD days)", fontsize=12.5)
axf.set_title("(c) Robustness of the mediated pathway",
              fontsize=15, fontweight="bold", pad=10, loc="left")
axf.set_ylim(-0.6, len(forest) - 0.4)

with mpl.rc_context({"savefig.bbox": None}):
    for ext in ("png", "pdf"):
        fig.savefig(FIGDIR / f"fig04_results_panel.{ext}")
    fig.savefig(
        FIGDIR / "fig04_results_panel.tif", format="tiff", dpi=300,
        pil_kwargs={"compression": "tiff_lzw"},
    )
print("  saved -> figures/fig04_results_panel.png | .pdf | .tif")
plt.close(fig)

# ---- Landmark-timing and delirium-definition sensitivities -----------------
# Re-derive the mediator (and landmark selection) at alternative landmarks and
# under a sustained-delirium definition, then rerun the full pipeline. Reported
# in Table 3 only (not the forest) to keep the figure legible.
def spec_landmark(land_h, sustained=False):
    def f(dfb):
        prev, inc, apre, apost, aever = derive_delirium(dfb, land_h, require_sustained=sustained)
        d = dfb.copy()
        d["M"] = inc.values
        d["in_landmark"] = (((aever == 1) & (prev == 0) & (apost == 1)).astype(int)).values
        return d
    return f

LAND_SPECS = [
    ("Landmark 24 h", spec_landmark(24)),
    ("Landmark 72 h", spec_landmark(72)),
    ("Sustained delirium (>=2 CAM)", spec_landmark(LAND_H, sustained=True)),
]
land_sens = {}
for j, (name, fn) in enumerate(LAND_SPECS):
    pt = _nie_point(fn, {"weight": "iptw"})
    lo, hi = _nie_ci(fn, {"weight": "iptw"}, RANDOM_STATE + 300 + j)
    land_sens[name] = (pt, (lo, hi))
    print(f"  {name:26s} NIE={pt:+.2f}  CI=({lo:+.2f}, {hi:+.2f})")

t3_rows = [
    ("Primary NIE (IPTW x selection, VFD days)", f"{est['NIE']:+.2f} ({ci['NIE'][0]:+.2f}, {ci['NIE'][1]:+.2f})"),
    ("Primary TE, broad cohort (VFD days)", f"{est['TE_broad']:+.2f} ({ci['TE_broad'][0]:+.2f}, {ci['TE_broad'][1]:+.2f})"),
    ("Proportion mediated", f"{est['PM']*100:.1f}%"),
]
for name, (pt, c) in list(sens.items()) + list(land_sens.items()):
    ci_txt = f" ({c[0]:+.2f}, {c[1]:+.2f})" if np.isfinite(c[0]) else ""
    t3_rows.append((f"NIE - {name}", f"{pt:+.2f}{ci_txt}"))
t3_rows += [
    ("Low-VFD RR (total, broad)", f"{rr_te:.2f} ({rr_lo:.2f}, {rr_hi:.2f})"),
    ("E-value (total, point)", f"{ev:.2f}"),
    ("E-value (total, CI limit)", f"{ev_ci:.2f}"),
    ("E-value (NIE, point)", f"{ev_nie:.2f}"),
    ("E-value (NIE, CI limit)", f"{ev_nie_ci:.2f}"),
]
table3 = pd.DataFrame(t3_rows, columns=["Analysis", "Value"])
table3.to_csv(TABDIR / "table3_sensitivity.csv", index=False)
print("Saved -> tables/table3_sensitivity.csv  and figures/fig04_results_panel.*")
table3





## 10.1 - Component associations and cause-specific hazards

Mediation papers report the building-block associations, not only the decomposition. Table 4 gives the weighted, Rubin-pooled A->M (odds ratio for incident delirium), M->Y and A->Y (adjusted mean differences in ventilator-free days). Because ventilator-free days is a continuous score, these are ratios/mean differences rather than hazards. Table 5 elevates the competing-risk endpoints to co-primary status: cause-specific Cox models for time to liberation from ventilation (with in-hospital death as a competing event, estimated for the treatment in the broad cohort and for delirium in the landmark cohort) and for time to death, so the mechanism is shown on a hazard scale and the competing risk of death is addressed directly. Table 6 adds the 28-day mortality risk ratio, ensuring the ventilator-free-days mediation is not driven solely by decedents scoring zero.


In [ ]:
# ============ Component associations + cause-specific hazards ==============
# Mediation reporting convention (Valeri-VanderWeele): alongside the natural-
# effect decomposition we report the constituent regression associations that
# build it - A->M, M->Y (adjusted for A) and A->Y - each IPTW-weighted and
# pooled across imputations with Rubin's rules.
#
# The primary outcome Y (ventilator-free days) is a continuous score, so A->Y
# and M->Y are reported as adjusted mean differences and A->M (binary delirium)
# as an odds ratio. Because "hazard" only applies to a time-to-event outcome, we
# ADD a secondary cause-specific hazard model for time to liberation from
# ventilation with in-hospital death treated as a competing event - this both
# provides a hazard-ratio framing and addresses the competing risk of death that
# the VFD score handles only by assigning 0.

def rubin_pool(betas, ses):
    betas = np.asarray(betas, float); ses = np.asarray(ses, float)
    m = len(betas)
    qbar = betas.mean()
    ubar = np.mean(ses ** 2)
    bvar = betas.var(ddof=1) if m > 1 else 0.0
    T = ubar + (1 + 1 / m) * bvar
    se = math.sqrt(T)
    return qbar, qbar - 1.96 * se, qbar + 1.96 * se

# ---- (1) Component associations (Rubin-pooled coefficients) ----------------
MED_NOINT = f"Y ~ A + M + {CTRL_FIT}"     # additive outcome model for clean M and A coefficients
TOTAL_F   = f"Y ~ A + {CTRL_FIT}"

am_b, am_s, my_b, my_s, ayd_b, ayd_s, ayt_b, ayt_s = ([] for _ in range(8))
for df in land_imputations:
    w = (df["sw"] / df["sw"].mean()).values
    mmod = smf.glm(MED_F, data=df, family=sm.families.Binomial(), var_weights=w).fit()
    ymod = smf.wls(MED_NOINT, data=df, weights=w).fit()
    tmod = smf.wls(TOTAL_F, data=df, weights=w).fit()
    am_b.append(mmod.params["A"]);  am_s.append(mmod.bse["A"])
    my_b.append(ymod.params["M"]);  my_s.append(ymod.bse["M"])
    ayd_b.append(ymod.params["A"]); ayd_s.append(ymod.bse["A"])
    ayt_b.append(tmod.params["A"]); ayt_s.append(tmod.bse["A"])

def _or_row(b, s, label):
    e, lo, hi = rubin_pool(b, s)
    return (label, f"{math.exp(e):.2f} ({math.exp(lo):.2f}, {math.exp(hi):.2f})")

def _md_row(b, s, label):
    e, lo, hi = rubin_pool(b, s)
    return (label, f"{e:+.2f} ({lo:+.2f}, {hi:+.2f})")

comp_rows = [
    _or_row(am_b, am_s, "A -> M: benzo -> incident delirium (OR)"),
    _md_row(my_b, my_s, "M -> Y: delirium -> VFD, adj. for A (mean diff, days)"),
    _md_row(ayd_b, ayd_s, "A -> Y: benzo -> VFD, adj. for M (mean diff, days)"),
    _md_row(ayt_b, ayt_s, "A -> Y: benzo -> VFD, total (mean diff, days)"),
]
table4 = pd.DataFrame(comp_rows, columns=["Association", "Estimate (95% CI)"])
table4.to_csv(TABDIR / "table4_components.csv", index=False)
print("Component associations (IPTW-weighted, Rubin-pooled):")
print(table4.to_string(index=False))

# ---- (2) Co-primary competing-risk and mortality endpoints -----------------
# These are elevated to co-primary mechanistic endpoints (not appendix
# components) so reviewers cannot attribute the VFD mediation solely to
# decedents scoring VFD = 0:
#   - A -> liberation from ventilation (cause-specific HR, death competing),
#     estimated in the BROAD cohort under IPTW (total effect on liberation);
#   - M -> liberation, adjusted for A, in the LANDMARK cohort under the
#     mediation weight (mechanistic);
#   - A -> 28-day death (cause-specific HR + risk ratio) in the BROAD cohort.
H = float(CONFIG["VFD_HORIZON_DAYS"])

def survival_frame(df, cause="liberation"):
    vent = df["vent_days_28d"].astype(float).clip(0, H).values
    dtd = df["days_to_death"].astype(float).values          # inf if alive through horizon
    dead = (df["dead_28d"].astype(int).values == 1)
    extubated = vent < H
    death_first = dead & (dtd <= vent)
    if cause == "liberation":
        event = (extubated & ~death_first).astype(int)
    else:                                                   # cause == "death"
        event = death_first.astype(int)
    T = np.where(extubated & ~death_first, vent,
                 np.where(death_first, np.minimum(dtd, H), H))
    T = np.clip(T, 1e-3, H)
    return pd.DataFrame({"T": T, "event": event,
                         "A": df["A"].astype(float).values,
                         "M": df["M"].astype(float).values,
                         "sw": (df["sw"] / df["sw"].mean()).values})

# 28-day mortality risk ratio (IPTW-standardized, broad cohort)
mort_b, mort_s = [], []
for dfb in imputations:
    w = (dfb["sw"] / dfb["sw"].mean()).values
    mm = smf.glm(f"dead_28d ~ A + {CTRL_FIT}", data=dfb, family=sm.families.Binomial(),
                 var_weights=w).fit()
    r1 = mm.predict(dfb.assign(A=1)).mean(); r0 = mm.predict(dfb.assign(A=0)).mean()
    mort_b.append(math.log(max(r1, 1e-9) / max(r0, 1e-9)))
    mort_s.append(float(mm.bse["A"]))
mrr, mlo, mhi = rubin_pool(mort_b, mort_s)
table6 = pd.DataFrame([
    ("A -> 28-day death (IPTW risk ratio, broad)",
     f"{math.exp(mrr):.2f} ({math.exp(mlo):.2f}, {math.exp(mhi):.2f})")],
    columns=["Association", "Estimate (95% CI)"])
table6.to_csv(TABDIR / "table6_mortality.csv", index=False)

try:
    from lifelines import CoxPHFitter
    ay_b, ay_s, myh_b, myh_s, dh_b, dh_s = [], [], [], [], [], []
    for dfb in imputations:                                 # A -> liberation / death (broad, IPTW)
        s_lib = survival_frame(dfb, "liberation")
        c1 = CoxPHFitter().fit(s_lib[["T", "event", "A", "sw"]], "T", "event",
                               weights_col="sw", robust=True)
        ay_b.append(c1.params_["A"]); ay_s.append(c1.standard_errors_["A"])
        s_die = survival_frame(dfb, "death")
        cd = CoxPHFitter().fit(s_die[["T", "event", "A", "sw"]], "T", "event",
                               weights_col="sw", robust=True)
        dh_b.append(cd.params_["A"]); dh_s.append(cd.standard_errors_["A"])
    for dl in land_imputations:                             # M -> liberation (landmark, mediation wt)
        s_lib = survival_frame(dl, "liberation")
        c2 = CoxPHFitter().fit(s_lib[["T", "event", "A", "M", "sw"]], "T", "event",
                               weights_col="sw", robust=True)
        myh_b.append(c2.params_["M"]); myh_s.append(c2.standard_errors_["M"])
    haz_rows = [
        _or_row(ay_b, ay_s, "A -> liberation: benzo (cause-specific HR, broad)"),
        _or_row(myh_b, myh_s, "M -> liberation: delirium, adj. for A (cause-specific HR)"),
        _or_row(dh_b, dh_s, "A -> death: benzo (cause-specific HR, broad)"),
    ]
    table5 = pd.DataFrame(haz_rows, columns=["Association", "Hazard ratio (95% CI)"])
    table5.to_csv(TABDIR / "table5_hazard.csv", index=False)
    print("\nCause-specific hazards (death as competing event) + mortality companion:")
    print(table5.to_string(index=False))
    print(table6.to_string(index=False))
    print("Liberation HR < 1 = slower liberation (worse); death HR > 1 = higher mortality.")
except Exception as exc:  # pragma: no cover
    print(f"\n[skipped cause-specific hazard table: {exc}]")
    print(table6.to_string(index=False))


## 11 - Summary, assumptions, and limitations

**Headline.** The confirmed finding is the total effect: early benzodiazepine sedation is associated with about 1.3 fewer ventilator-free days (broad cohort). Incident delirium is a small, assumption-sensitive pathway (proportion mediated ~7%; indirect-effect CI crosses zero; NIE E-value at the CI limit ~1.0), so we present delirium mediation as a secondary mechanism rather than the primary claim. Several sensitivities still show a negative indirect effect (midazolam-only, exclude indication, sustained delirium), so the pathway is not dismissed, only downgraded from a confident conclusion.

**What we estimated.** Following a pre-specified target-trial protocol, we emulated a trial of early benzodiazepine versus non-benzodiazepine continuous sedation in first adult ICU stays with early invasive ventilation. The total effect on ventilator-free days is estimated in the broad early-eligible cohort; within a 48 h landmark cohort we decomposed the effect into direct and indirect components through incident CAM-ICU delirium, reweighting the landmark sample back to the broad population with selection weights.

**Key design strengths.**
- A written target-trial protocol (Table 0) with an explicit time zero, landmark, censoring, competing events, causal contrasts, and a frozen primary estimand separated from exploratory analyses.
- Clear separation of time zero (exposure, 0-24 h) from the landmark (mediator, after 48 h) and outcome (to day 28); the landmark is a mediation restriction only, and the total effect is reported in the full early-eligible cohort.
- Inverse-probability-of-selection weights for remaining in ICU and being CAM-assessable past the landmark, so estimates generalize to the broad population rather than to landmark survivors alone.
- A confounder set targeting confounding by indication (alcohol/sedative/seizure flag, comorbidity burden, ICU unit, admission era, early opioids) plus a measured SOFA-lite severity score, entered in the propensity model AND symmetrically in the mediator and outcome models to control mediator-outcome confounding.
- Both natural and interventional direct/indirect effects, the latter for when cross-world assumptions are untenable in ICU data.
- Two estimators for the decomposition: weighted g-computation and an inverse-odds-ratio mediator-weighting (IORW) estimator that handles the mediator with weights rather than an outcome model, reported side by side.
- Overlap (ATO) weighting reported co-primary with IPTW: it achieves exact mean balance on every propensity covariate by construction, so the Love-plot residuals under IPTW are bounded by a weighting scheme with guaranteed balance.
- Competing-risk cause-specific hazards for liberation and death, and a 28-day mortality companion, elevated as co-primary mechanistic endpoints so the VFD mediation is not an artifact of decedents scoring VFD = 0.
- Multiple-imputation inference by Rubin's rules (per-imputation bootstrap re-estimating treatment and selection weights each draw + between-imputation variance); E-values for the total effect AND the natural indirect effect (point and CI limit).
- A sensitivity hierarchy: midazolam-only, no deep sedation, exclude hard indications, exclude CVICU, overlap weights, unweighted, landmark 24/48/72 h, and sustained (>= 2 positive CAM) delirium.

**Assumptions and limitations.** Natural-effect identification still requires no unmeasured confounding of A-Y, A-M, and M-Y, and no A-affected mediator-outcome confounder; interventional effects relax the cross-world part but not the confounding assumptions, and post-treatment sedation depth is a plausible intermediate confounder probed only in sensitivity. Selection weights assume the landmark-inclusion model is correctly specified. The SOFA-lite score omits the respiratory (PaO2/FiO2) component, unavailable in this extract. Comorbidity and indication flags come from admission ICD codes, which span the hospitalization and may include conditions recognized after ICU admission. CAM-ICU assessability depends on sedation depth, so landmark inclusion is potentially differential by exposure despite the selection weights. VFD assigns 0 to early deaths by design. Single-center academic data (MIMIC-IV).


In [ ]:
import platform
print("SESSION INFO"); print("-" * 40)
print("date     :", pd.Timestamp.now())
print("platform :", platform.platform())
print("python   :", sys.version.split()[0])
print("M_IMPUT  :", M_IMPUT)
print("N analytic stays:", len(dat))
print("Figures:", sorted(p.name for p in FIGDIR.glob("*.png")))
print("Tables :", sorted(p.name for p in TABDIR.glob("*.csv")))
print("Expected key figures: fig00_dag, fig01_flow, fig02_descriptive,")
print("  fig03_positivity_balance, fig04_results_panel")
